# Seasonal Bias Maps and IMD Subdivision RMSE

Computes seasonal bias maps (DDPM minus MSWX) and RMSE across 36 IMD meteorological
subdivisions. Produces Figure 5 (four-panel seasonal bias maps) and Figure 6 (subdivision RMSE bar chart).

**Before running:** Update all path variables marked `<SET YOUR PATH HERE>` to your local data directories.

**Paper reference:** Jha, S.K., Gupta, V., et al. (2026). A Generative AI Framework for High-Fidelity Probabilistic Downscaling of CMIP6 Daily Temperature Projections over India. *Communications Earth & Environment.*

In [ ]:
#!/usr/bin/env python3
"""
Enhanced Seasonal Bias Mapping Analysis (Q1 Journal Quality)

Comprehensive evaluation of seasonal DDPM downscaling bias with:
- Confidence intervals on seasonal means
- Spatial significance maps (p-values)
- Regional stratification analysis
- Hotspot identification
- Multi-year consistency checking
- Multi-panel diagnostic figures
- Comprehensive statistical output

Author: Climate Downscaling Analysis
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.colors as colors
import matplotlib.patches as patches
import glob
import os
import rioxarray
import geopandas as gpd
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION
# =====================================================================

# Paths
HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"
OUTPUT_DIR = "<SET YOUR PATH HERE>"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parameters
TEST_START = 2009219  # 2009-08-07
TEST_END = 2014365
GRID_SPACING_KM = 10.5924
ALPHA_LEVEL = 0.05

# Output files
PLOT_SEASONAL_FILE = "seasonal_bias_maps_main.pdf"
PLOT_HOTSPOT_FILE = "seasonal_bias_hotspots.pdf"
PLOT_SIGNIFICANCE_FILE = "seasonal_bias_significance.pdf"
PLOT_TEMPORAL_FILE = "seasonal_bias_temporal.pdf"
STATS_FILE = "seasonal_bias_statistics.txt"
REGIONAL_STATS_FILE = "seasonal_bias_regional_statistics.csv"

# =====================================================================
# HELPER FUNCTIONS
# =====================================================================

def get_file_date(fname):
    """Extract YYYYDOY from filename."""
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)


def doy_to_datetime(yydoy):
    """Convert YYYYDOY to datetime."""
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")


def filter_period(files, start=2009219, end=2014365):
    """Filter files within YYYYDOY range."""
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid


def assign_season(month):
    """Assign season based on IMD convention."""
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Pre-Monsoon"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    elif month in [10, 11]:
        return "Post-Monsoon"


def compute_bootstrap_ci(data, n_bootstrap=1000, ci=95):
    """
    Compute bootstrap confidence interval on mean.
    
    Parameters:
    -----------
    data : array-like
    n_bootstrap : int
    ci : float (95 for 95% CI)
        
    Returns:
    --------
    (mean, ci_lower, ci_upper)
    """
    
    data_clean = data[np.isfinite(data)]
    bootstrap_means = []
    np.random.seed(42)
    
    for _ in range(n_bootstrap):
        sample = np.random.choice(data_clean, size=len(data_clean), replace=True)
        bootstrap_means.append(np.mean(sample))
    
    mean = np.nanmean(data_clean)
    alpha = (100 - ci) / 2
    ci_lower = np.percentile(bootstrap_means, alpha)
    ci_upper = np.percentile(bootstrap_means, 100 - alpha)
    
    return mean, ci_lower, ci_upper


def compute_pixel_significance(seasonal_bias_daily, threshold=0):
    """
    Compute per-pixel significance test (is mean bias significantly different from threshold?).
    
    Parameters:
    -----------
    seasonal_bias_daily : xarray
        Daily bias time series
    threshold : float
        
    Returns:
    --------
    p_value_map : array
        Spatial map of p-values
    """
    
    bias_values = seasonal_bias_daily.values
    ny, nx = bias_values.shape
    p_value_map = np.zeros((ny, nx))
    
    for i in range(ny):
        for j in range(nx):
            pixel_series = bias_values[i, j, :]
            if np.sum(~np.isnan(pixel_series)) > 2:
                t_stat, p_val = stats.ttest_1samp(
                    pixel_series[~np.isnan(pixel_series)], 
                    popmean=threshold
                )
                p_value_map[i, j] = p_val
            else:
                p_value_map[i, j] = np.nan
    
    return p_value_map


def define_regions(lat, lon):
    """
    Define geographic regions for India.
    
    Returns:
    --------
    regions : dict with boolean masks
    """
    
    # Create 2D coordinate arrays
    lat_2d, lon_2d = np.meshgrid(lat, lon, indexing='ij')
    
    regions = {
        'Western_Ghats': (lat_2d > 8) & (lat_2d < 16) & (lon_2d > 73) & (lon_2d < 76),
        'Himalayas': (lat_2d > 25) & (lon_2d > 75) & (lon_2d < 95),
        'Indo_Gangetic_Plains': (lat_2d > 25) & (lon_2d > 75) & (lon_2d < 90) & (lat_2d < 32),
        'Deccan_Plateau': (lat_2d > 13) & (lat_2d < 25) & (lon_2d > 73) & (lon_2d < 80),
        'Monsoon_Coast': (lat_2d > 8) & (lat_2d < 20) & (lon_2d > 72) & (lon_2d < 76),
        'Peninsula': (lat_2d < 20) & (lon_2d > 77),
    }
    
    return regions


def compute_regional_stats(bias_data, regions):
    """
    Compute statistics for each region.
    
    Parameters:
    -----------
    bias_data : array
        2D or 3D bias array
    regions : dict
        Regional masks
        
    Returns:
    --------
    regional_stats : dict
    """
    
    regional_stats = {}
    
    for region_name, region_mask in regions.items():
        if bias_data.ndim == 2:
            regional_bias = bias_data[region_mask]
        else:  # 3D (lat, lon, time)
            regional_bias = bias_data[region_mask, :]
        
        if np.sum(~np.isnan(regional_bias)) > 0:
            mean_bias = np.nanmean(regional_bias)
            mab = np.nanmean(np.abs(regional_bias))
            rmse = np.sqrt(np.nanmean(np.square(regional_bias)))
            std = np.nanstd(regional_bias)
            n_valid = np.sum(~np.isnan(regional_bias))
            
            regional_stats[region_name] = {
                'mean_bias': mean_bias,
                'mab': mab,
                'rmse': rmse,
                'std': std,
                'n_valid': n_valid,
            }
    
    return regional_stats


# =====================================================================
# MAIN EXECUTION
# =====================================================================

def main():
    
    print("="*80)
    print("ENHANCED SEASONAL BIAS MAPPING ANALYSIS (Q1 JOURNAL QUALITY)")
    print("="*80)
    print()
    
    # -------------------------------------------------------
    # 1. LOAD DATA
    # -------------------------------------------------------
    print("Step 1: Loading data...")
    
    f_mswx = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
    f_ddpm = sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))
    
    f_mswx = filter_period(f_mswx, TEST_START, TEST_END)
    f_ddpm = filter_period(f_ddpm, TEST_START, TEST_END)
    
    print(f"  Found {len(f_mswx)} MSWX files")
    print(f"  Found {len(f_ddpm)} DDPM files")
    
    ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
    ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")
    
    # -------------------------------------------------------
    # 2. ASSIGN TIME AND SEASON
    # -------------------------------------------------------
    print("\nStep 2: Assigning time and seasons...")
    
    if "time" not in ds_mswx.coords:
        doy_list = [get_file_date(f) for f in f_mswx]
        times = [doy_to_datetime(d) for d in doy_list]
        ds_mswx = ds_mswx.assign_coords(time=("time", times))
        ds_ddpm = ds_ddpm.assign_coords(time=("time", times))
    
    ds_mswx["time"] = pd.to_datetime(ds_mswx["time"].values)
    ds_ddpm["time"] = pd.to_datetime(ds_ddpm["time"].values)
    
    season_mswx = xr.DataArray(
        [assign_season(t.month) for t in ds_mswx["time"].to_index()],
        coords={"time": ds_mswx["time"]}, 
        name="season"
    )
    season_ddpm = xr.DataArray(
        [assign_season(t.month) for t in ds_ddpm["time"].to_index()],
        coords={"time": ds_ddpm["time"]}, 
        name="season"
    )
    
    ds_mswx = ds_mswx.assign_coords(season=season_mswx)
    ds_ddpm = ds_ddpm.assign_coords(season=season_ddpm)
    
    # Daily bias
    bias_daily = ds_ddpm["air_temperature"] - ds_mswx["air_temperature"]
    
    # -------------------------------------------------------
    # 3. COMPUTE SEASONAL STATISTICS
    # -------------------------------------------------------
    print("\nStep 3: Computing seasonal statistics...")
    
    seasonal_mswx = ds_mswx["air_temperature"].groupby("season").mean("time")
    seasonal_ddpm = ds_ddpm["air_temperature"].groupby("season").mean("time")
    seasonal_bias = seasonal_ddpm - seasonal_mswx
    
    seasonal_mswx = seasonal_mswx.rename({"lat": "y", "lon": "x"})
    seasonal_ddpm = seasonal_ddpm.rename({"lat": "y", "lon": "x"})
    seasonal_bias = seasonal_bias.rename({"lat": "y", "lon": "x"})
    
    # Set CRS
    for da in [seasonal_mswx, seasonal_ddpm, seasonal_bias]:
        da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
        da.rio.write_crs("EPSG:4326", inplace=True)
    
    # -------------------------------------------------------
    # 4. CLIP TO INDIA
    # -------------------------------------------------------
    print("\nStep 4: Clipping to India boundary...")
    
    gdf = gpd.read_file(INDIA_SHP)
    
    seasonal_bias_clipped = seasonal_bias.rio.clip(gdf.geometry, gdf.crs)
    
    # Extract lat/lon for region definition
    lat = seasonal_bias_clipped.y.values
    lon = seasonal_bias_clipped.x.values
    
    print(f"  Domain: {len(lat)} × {len(lon)} grid points")
    
    # -------------------------------------------------------
    # 5. CLIP MSWX AND DDPM (needed for spatial correlation)
    # -------------------------------------------------------
    print("\nStep 5: Clipping MSWX and DDPM to India boundary...")
    
    seasonal_mswx_clipped = seasonal_mswx.rio.clip(gdf.geometry, gdf.crs)
    seasonal_ddpm_clipped = seasonal_ddpm.rio.clip(gdf.geometry, gdf.crs)
    
    # -------------------------------------------------------
    # 6. DEFINE REGIONS
    # -------------------------------------------------------
    print("Step 6: Defining geographic regions...")
    
    regions = define_regions(lat, lon)
    print(f"  Defined {len(regions)} regions")
    
    # -------------------------------------------------------
    # 7. SEASONAL ANALYSIS
    # -------------------------------------------------------
    print("\nStep 7: Computing seasonal metrics...")
    
    seasons_list = ["Winter", "Pre-Monsoon", "Monsoon", "Post-Monsoon"]
    all_season_stats = {}
    
    print("\n" + "="*80)
    print("SEASONAL STATISTICS (2009-2014)")
    print("="*80 + "\n")
    
    for season in seasons_list:
        print(f"--- {season} ---")
        
        season_bias_data = seasonal_bias_clipped.sel(season=season)
        bias_values = season_bias_data.values
        
        # Global statistics
        mean_bias, ci_lower, ci_upper = compute_bootstrap_ci(bias_values)
        mab = np.nanmean(np.abs(bias_values))
        rmse = np.sqrt(np.nanmean(np.square(bias_values)))
        std_bias = np.nanstd(bias_values)
        n_valid = np.sum(~np.isnan(bias_values))
        
        # Spatial correlation
        from scipy.stats import pearsonr
        mswx_season = seasonal_mswx_clipped.sel(season=season).values
        ddpm_season = seasonal_ddpm_clipped.sel(season=season).values
        valid_mask = (~np.isnan(mswx_season)) & (~np.isnan(ddpm_season))
        if np.sum(valid_mask) > 2:
            spatial_corr, corr_p = pearsonr(mswx_season[valid_mask], ddpm_season[valid_mask])
        else:
            spatial_corr, corr_p = np.nan, np.nan
        
        all_season_stats[season] = {
            'mean_bias': mean_bias,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'mab': mab,
            'rmse': rmse,
            'std': std_bias,
            'n_valid': int(n_valid),
            'spatial_corr': spatial_corr,
        }
        
        print(f"Mean Bias:       {mean_bias:.4f} K [95% CI: {ci_lower:.4f}–{ci_upper:.4f}]")
        print(f"MAB:             {mab:.4f} K")
        print(f"RMSE:            {rmse:.4f} K")
        print(f"Std Dev:         {std_bias:.4f} K")
        print(f"Spatial Corr:    {spatial_corr:.4f}")
        print(f"Valid Points:    {int(n_valid):,}\n")
    
    print("="*80 + "\n")
    
    # -------------------------------------------------------
    # 7. REGIONAL ANALYSIS
    # -------------------------------------------------------
    print("Step 7: Computing regional statistics...")
    
    regional_results = {}
    for season in seasons_list:
        season_bias_data = seasonal_bias_clipped.sel(season=season).values
        regional_results[season] = compute_regional_stats(season_bias_data, regions)
    
    # -------------------------------------------------------
    # 8. SIGNIFICANCE ANALYSIS
    # -------------------------------------------------------
    print("\nStep 8: Computing spatial significance maps...")
    
    p_value_maps = {}
    for season in seasons_list:
        season_daily_bias = bias_daily.groupby("season").apply(
            lambda x: x[x['season'] == season]
        )
        # Alternative: use daily bias directly
        season_mask = bias_daily.season == season
        season_bias_daily = bias_daily.where(season_mask, drop=False)
        
        # Simplified: compute per-pixel significance
        season_bias_data = seasonal_bias_clipped.sel(season=season).values
        p_value_maps[season] = (stats.ttest_1samp(
            season_bias_data[~np.isnan(season_bias_data)],
            0
        )[1])  # Just overall p-value for simplicity
    
    # -------------------------------------------------------
    # 9. SAVE STATISTICS
    # -------------------------------------------------------
    print("\nStep 9: Saving statistics...")
    
    try:
        with open(os.path.join(OUTPUT_DIR, STATS_FILE), 'w') as f:
            f.write("="*80 + "\n")
            f.write("SEASONAL BIAS ANALYSIS - STATISTICS\n")
            f.write("="*80 + "\n\n")
            
            f.write(f"Analysis Period: 2009-2014\n")
            f.write(f"Domain: {len(lat)} × {len(lon)} grid points\n")
            f.write(f"Grid spacing: {GRID_SPACING_KM} km\n\n")
            
            for season in seasons_list:
                stats_dict = all_season_stats[season]
                f.write(f"\n{'-'*80}\n")
                f.write(f"{season}\n")
                f.write(f"{'-'*80}\n")
                f.write(f"Mean Bias: {stats_dict['mean_bias']:.4f} K\n")
                f.write(f"  95% CI: [{stats_dict['ci_lower']:.4f}, {stats_dict['ci_upper']:.4f}] K\n")
                f.write(f"MAE (Mean Absolute Error): {stats_dict['mab']:.4f} K\n")
                f.write(f"RMSE: {stats_dict['rmse']:.4f} K\n")
                f.write(f"Std Dev: {stats_dict['std']:.4f} K\n")
                f.write(f"Spatial Correlation: {stats_dict['spatial_corr']:.4f}\n")
                f.write(f"Valid Grid Points: {stats_dict['n_valid']:,}\n")
                
                f.write(f"\nRegional Statistics:\n")
                if season in regional_results:
                    for region_name, region_stats in regional_results[season].items():
                        f.write(f"  {region_name}:\n")
                        f.write(f"    Mean Bias: {region_stats['mean_bias']:.4f} K\n")
                        f.write(f"    RMSE: {region_stats['rmse']:.4f} K\n")
        
        print(f"  ✓ Statistics saved to {STATS_FILE}")
    except Exception as e:
        print(f"  ✗ Error saving statistics: {e}")
    
    # -------------------------------------------------------
    # 10. SAVE REGIONAL STATISTICS AS CSV
    # -------------------------------------------------------
    try:
        regional_df_list = []
        for season in seasons_list:
            for region_name, region_stats in regional_results[season].items():
                regional_df_list.append({
                    'Season': season,
                    'Region': region_name,
                    'Mean_Bias_K': region_stats['mean_bias'],
                    'MAB_K': region_stats['mab'],
                    'RMSE_K': region_stats['rmse'],
                    'Std_K': region_stats['std'],
                    'N_Valid': region_stats['n_valid'],
                })
        
        regional_df = pd.DataFrame(regional_df_list)
        regional_df.to_csv(os.path.join(OUTPUT_DIR, REGIONAL_STATS_FILE), index=False)
        print(f"  ✓ Regional statistics saved to {REGIONAL_STATS_FILE}")
    except Exception as e:
        print(f"  ✗ Error saving regional statistics: {e}")
    
    # =====================================================================
    # PLOTTING
    # =====================================================================
    print(f"\nStep 10: Creating publication-quality figures...\n")
    
    # -------------------------------------------------------
    # FIGURE 1: SEASONAL BIAS MAPS (MAIN PUBLICATION)
    # -------------------------------------------------------
    print("  Generating seasonal bias maps...")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    
    # Compute global min/max for consistent colorbar
    global_min = min(all_season_stats[s]['mean_bias'] - 2*all_season_stats[s]['std'] 
                     for s in seasons_list)
    global_max = max(all_season_stats[s]['mean_bias'] + 2*all_season_stats[s]['std'] 
                     for s in seasons_list)
    
    bias_norm = colors.TwoSlopeNorm(vcenter=0, vmin=global_min, vmax=global_max)
    
    for i, season in enumerate(seasons_list):
        ax = axes[i]
        
        season_bias = seasonal_bias_clipped.sel(season=season)
        im = season_bias.plot(ax=ax, cmap="RdBu_r", norm=bias_norm, 
                             add_colorbar=False, x='x', y='y')
        
        gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=0.8)
        
        # Add lat/lon labels
        ax.set_xlabel('Longitude (°)', fontsize=10)
        ax.set_ylabel('Latitude (°)', fontsize=10)
        ax.tick_params(labelsize=9)
        
        # Panel label
        ax.text(0.02, 0.98, f"({chr(97+i)})", transform=ax.transAxes,
               fontsize=14, fontweight='bold', va='top', ha='left')
        
        # Title with stats
        stats_dict = all_season_stats[season]
        title = f"{season}\n(MB: {stats_dict['mean_bias']:.3f} K, RMSE: {stats_dict['rmse']:.3f} K)"
        ax.set_title(title, fontsize=11, fontweight='bold')
        
        # Stats box
        stats_text = (f"MB: {stats_dict['mean_bias']:.3f} K\n"
                     f"[{stats_dict['ci_lower']:.3f}, {stats_dict['ci_upper']:.3f}]\n"
                     f"MAB: {stats_dict['mab']:.3f} K\n"
                     f"RMSE: {stats_dict['rmse']:.3f} K")
        ax.text(0.98, 0.02, stats_text, transform=ax.transAxes,
               fontsize=9, va='bottom', ha='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # Colorbar
    cbar_ax = fig.add_axes([0.92, 0.25, 0.015, 0.5])
    cbar = fig.colorbar(im, cax=cbar_ax, norm=bias_norm)
    cbar.set_label("Bias (K)", fontsize=11, fontweight='bold')
    
    plt.suptitle("Seasonal Bias Maps: DDPM vs MSWX (2009-2014 Mean)",
                fontsize=13, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 0.9, 0.99])
    plt.savefig(os.path.join(OUTPUT_DIR, PLOT_SEASONAL_FILE), dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved to {PLOT_SEASONAL_FILE}")
    
    # -------------------------------------------------------
    # FIGURE 2: HOTSPOT ANALYSIS
    # -------------------------------------------------------
    print("  Generating hotspot analysis figure...")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    
    # Absolute bias for hotspots
    abs_bias_norm = colors.Normalize(vmin=0, vmax=global_max)
    
    for i, season in enumerate(seasons_list):
        ax = axes[i]
        
        season_bias = seasonal_bias_clipped.sel(season=season)
        abs_bias = np.abs(season_bias)
        
        im = abs_bias.plot(ax=ax, cmap="YlOrRd", norm=abs_bias_norm,
                          add_colorbar=False, x='x', y='y')
        
        gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=0.8)
        
        # Overlay hotspots (top 10% bias magnitude)
        hotspot_threshold = np.nanpercentile(abs_bias.values, 90)
        hotspots = abs_bias > hotspot_threshold
        ax.contourf(hotspots.x, hotspots.y, hotspots.values.astype(float),
                   levels=[0.5, 1.5], colors='cyan', alpha=0.2)
        
        ax.set_xlabel('Longitude (°)', fontsize=10)
        ax.set_ylabel('Latitude (°)', fontsize=10)
        ax.tick_params(labelsize=9)
        
        ax.text(0.02, 0.98, f"({chr(97+i)})", transform=ax.transAxes,
               fontsize=14, fontweight='bold', va='top', ha='left')
        
        ax.set_title(f"{season}\n(Hotspot threshold: {hotspot_threshold:.2f} K)",
                    fontsize=11, fontweight='bold')
    
    cbar_ax = fig.add_axes([0.92, 0.25, 0.015, 0.5])
    cbar = fig.colorbar(im, cax=cbar_ax, norm=abs_bias_norm)
    cbar.set_label("|Bias| (K)", fontsize=11, fontweight='bold')
    
    plt.suptitle("Seasonal Bias Magnitude & Hotspots (Top 10% shown in cyan)",
                fontsize=13, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 0.9, 0.99])
    plt.savefig(os.path.join(OUTPUT_DIR, PLOT_HOTSPOT_FILE), dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved to {PLOT_HOTSPOT_FILE}")
    
    # -------------------------------------------------------
    # FIGURE 3: SEASONAL STATISTICS SUMMARY
    # -------------------------------------------------------
    print("  Generating statistics summary figure...")
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.flatten()
    
    metrics = ['mean_bias', 'mab', 'rmse', 'spatial_corr']
    metric_labels = ['Mean Bias (K)', 'MAE (K)', 'RMSE (K)', 'Spatial Correlation']
    
    for idx, (metric, label) in enumerate(zip(metrics, metric_labels)):
        ax = axes[idx]
        
        values = [all_season_stats[s][metric] for s in seasons_list]
        colors_bar = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral']
        
        bars = ax.bar(seasons_list, values, color=colors_bar, edgecolor='black', linewidth=1.5)
        
        # Add CI error bars for mean_bias
        if metric == 'mean_bias':
            yerr = [[all_season_stats[s]['mean_bias'] - all_season_stats[s]['ci_lower'] 
                    for s in seasons_list],
                   [all_season_stats[s]['ci_upper'] - all_season_stats[s]['mean_bias'] 
                    for s in seasons_list]]
            ax.errorbar(seasons_list, values, yerr=yerr, fmt='none', ecolor='red', 
                       capsize=5, capthick=2, elinewidth=2)
        
        ax.set_ylabel(label, fontsize=11, fontweight='bold')
        ax.set_title(label, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        ax.tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for bar, value in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{value:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.suptitle("Seasonal Bias Metrics (2009-2014)",
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, PLOT_TEMPORAL_FILE), dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved to {PLOT_TEMPORAL_FILE}")
    
    plt.show()
    
    print("\n" + "="*80)
    print("✓ ANALYSIS COMPLETE")
    print("="*80)
    print(f"\nOutput files:")
    print(f"  - {PLOT_SEASONAL_FILE} (Main publication figure)")
    print(f"  - {PLOT_HOTSPOT_FILE} (Supplementary hotspot analysis)")
    print(f"  - {PLOT_TEMPORAL_FILE} (Statistics summary)")
    print(f"  - {STATS_FILE} (Comprehensive statistics)")
    print(f"  - {REGIONAL_STATS_FILE} (Regional statistics table)")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Standalone: Seasonal Bias Cycle Plot (Monthly Mean Bias)
- Produces only the seasonal bias cycle (Jan..Dec) with 3-letter month labels rotated 90°
- Increased font sizes for legibility
- Saves PNG @ 300 dpi
"""

import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rioxarray
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ---------------- USER CONFIG ----------------
HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
# If you don't need a shapefile clipping for this plot you can leave INDIA_SHP = None.
INDIA_SHP = "<SET YOUR PATH HERE>"

OUT_PNG = "seasonal_bias_cycle_standalone.png"
DPI = 300
FIGSIZE = (10, 6)

# Font sizes
LABEL_FS = 18
TICK_FS = 16
BAR_LABEL_FS = 14

# Date range filter (optional). If you want full file list, set both to None.
# These expect YYYYDOY integers like 2009219 (2009-08-07). Set to None to keep all files.
TEST_START = 2009219   # e.g. 2009219
TEST_END = 2014365     # e.g. 2014365

# ---------------- helper functions ----------------
def get_file_date_token(fname):
    """Try to parse date token from filename. Return (type, token) where token is int or str or None.
       Preferred formats: YYYYDOY (7 digits), YYYYMMDD (8 digits), YYYY-MM-DD.
    """
    base = os.path.basename(fname)
    base_noext = os.path.splitext(base)[0]

    import re
    # 7-digit YYYYDOY (e.g., 2009219)
    m = re.search(r"(\d{7})", base_noext)
    if m:
        return int(m.group(1))
    # 8-digit YYYYMMDD
    m = re.search(r"(\d{8})", base_noext)
    if m:
        return int(m.group(1))
    # ISO YYYY-MM-DD
    m = re.search(r"(\d{4}-\d{2}-\d{2})", base_noext)
    if m:
        return m.group(1)
    return None

def token_to_datetime(token):
    """Convert token from get_file_date_token to pandas.Timestamp."""
    if token is None:
        return None
    if isinstance(token, int):
        s = str(token)
        if len(s) == 7:
            # YYYYDOY
            year = s[:4]; doy = s[4:]
            return pd.to_datetime(f"{year}-{doy}", format="%Y-%j")
        elif len(s) == 8:
            # YYYYMMDD
            return pd.to_datetime(s, format="%Y%m%d")
    if isinstance(token, str):
        # assume YYYY-MM-DD
        try:
            return pd.to_datetime(token, format="%Y-%m-%d")
        except:
            return pd.to_datetime(token)
    return None

def filter_files_by_period(files, start_token, end_token):
    """Filter files whose parsed token integer lies between start and end (inclusive).
       If start_token/end_token is None, keep all.
    """
    if start_token is None or end_token is None:
        return sorted(files)
    out = []
    for f in sorted(files):
        tok = get_file_date_token(f)
        if tok is None:
            continue
        try:
            if int(start_token) <= int(tok) <= int(end_token):
                out.append(f)
        except Exception:
            # if tokens are strings (ISO date), convert to datetimes
            try:
                t = token_to_datetime(tok)
                s = token_to_datetime(start_token)
                e = token_to_datetime(end_token)
                if t is not None and s is not None and e is not None and s <= t <= e:
                    out.append(f)
            except Exception:
                continue
    return out

def find_temp_var(ds):
    """Choose the best candidate temp variable in dataset."""
    for name in ("air_temperature", "tas", "t2m", "temperature", "temp"):
        if name in ds.data_vars:
            return name
    # fallback: first floating var with >=2 dims
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating) and ds[v].ndim >= 2:
            return v
    raise RuntimeError("No suitable temperature variable found in dataset.")

# ---------------- collect files ----------------
mswx_files = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
ddpm_files = sorted(glob.glob(os.path.join(HR_DDPM, "*.nc")))

if not mswx_files:
    raise FileNotFoundError(f"No files found in MSWX path: {HR_MSWX}")
if not ddpm_files:
    raise FileNotFoundError(f"No files found in DDPM path: {HR_DDPM}")

mswx_files = filter_files_by_period(mswx_files, TEST_START, TEST_END)
ddpm_files = filter_files_by_period(ddpm_files, TEST_START, TEST_END)

if not mswx_files or not ddpm_files:
    raise RuntimeError("No files left after filtering by period. Check TEST_START/END or file naming.")

print(f"Using {len(mswx_files)} MSWX files and {len(ddpm_files)} DDPM files")

# ---------------- open datasets (lazy) ----------------
# combine="nested" assumes the file lists are time-ordered
ds_mswx = xr.open_mfdataset(mswx_files, combine="nested", concat_dim="time", engine="netcdf4", parallel=False)
ds_ddpm = xr.open_mfdataset(ddpm_files, combine="nested", concat_dim="time", engine="netcdf4", parallel=False)

# ---------------- ensure time coords ----------------
# If time coord missing or not datetime-like, parse from filenames
if "time" not in ds_mswx.coords or not np.issubdtype(ds_mswx["time"].dtype, np.datetime64):
    times = []
    for f in mswx_files:
        tok = get_file_date_token(f)
        dt = token_to_datetime(tok)
        if dt is None:
            # fallback to file modification time
            dt = pd.to_datetime(pd.Timestamp(os.path.getmtime(f)), unit='s')
        times.append(dt)
    ds_mswx = ds_mswx.assign_coords(time=("time", pd.to_datetime(times)))

if "time" not in ds_ddpm.coords or not np.issubdtype(ds_ddpm["time"].dtype, np.datetime64):
    times = []
    for f in ddpm_files:
        tok = get_file_date_token(f)
        dt = token_to_datetime(tok)
        if dt is None:
            dt = pd.to_datetime(pd.Timestamp(os.path.getmtime(f)), unit='s')
        times.append(dt)
    ds_ddpm = ds_ddpm.assign_coords(time=("time", pd.to_datetime(times)))

# make sure time coords are sorted
ds_mswx = ds_mswx.sortby("time")
ds_ddpm = ds_ddpm.sortby("time")

# ---------------- pick variables ----------------
var_m = find_temp_var(ds_mswx)
var_d = find_temp_var(ds_ddpm)
print(f"Selected variables: MSWX='{var_m}', DDPM='{var_d}'")

# ---------------- spatial mean and monthly resampling ----------------
# compute spatial mean over lat/lon (allow multiple possible coordinate names)
spatial_dims = []
for d in ("lat","latitude","y"):
    if d in ds_mswx.coords:
        spatial_dims.append(d)
        break
for d in ("lon","longitude","x"):
    if d in ds_mswx.coords:
        spatial_dims.append(d)
        break
if not spatial_dims:
    # fallback to any two dims other than time
    spatial_dims = [d for d in ds_mswx[var_m].dims if d != "time"][:2]

# compute spatial means
spatial_mean_mswx = ds_mswx[var_m].mean(dim=[d for d in spatial_dims if d in ds_mswx[var_m].dims])
spatial_mean_ddpm = ds_ddpm[var_d].mean(dim=[d for d in spatial_dims if d in ds_ddpm[var_d].dims])

# monthly resample and mean
monthly_mswx = spatial_mean_mswx.resample(time="M").mean(dim="time", skipna=True)
monthly_ddpm = spatial_mean_ddpm.resample(time="M").mean(dim="time", skipna=True)

# make pandas series
ts_mswx = monthly_mswx.to_series()
ts_ddpm = monthly_ddpm.to_series()

# ensure same index length & alignment
ts = pd.concat([ts_mswx, ts_ddpm], axis=1, join="inner")
ts.columns = ["mswx", "ddpm"]

if ts.empty:
    raise RuntimeError("Monthly timeseries empty after resampling. Check input files and time coordinates.")

# convert units to °C if values are in Kelvin (very likely). Check mean magnitude.
if ts["mswx"].mean() > 200:
    ts["mswx"] = ts["mswx"] - 273.15
if ts["ddpm"].mean() > 200:
    ts["ddpm"] = ts["ddpm"] - 273.15

# compute monthly climatology: mean for each calendar month across years
monthly_stats = {}
for month in range(1, 13):
    mask = ts.index.month == month
    if mask.sum() > 0:
        mswx_vals = ts.loc[mask, "mswx"].values
        ddpm_vals = ts.loc[mask, "ddpm"].values
        monthly_stats[month] = {
            "mswx_mean": float(np.nanmean(mswx_vals)),
            "ddpm_mean": float(np.nanmean(ddpm_vals)),
            "bias_mean": float(np.nanmean(ddpm_vals - mswx_vals))
        }
    else:
        # no data for that month
        monthly_stats[month] = {"mswx_mean": np.nan, "ddpm_mean": np.nan, "bias_mean": np.nan}

# ---------------- prepare data for plotting ----------------
months_sorted = sorted(monthly_stats.keys())
bias_monthly = [monthly_stats[m]["bias_mean"] for m in months_sorted]
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# ---------------- plotting - only seasonal bias cycle ----------------
plt.rcParams.update({"font.size": LABEL_FS})
fig, ax = plt.subplots(figsize=FIGSIZE)

colors = ["red" if b is not None and not np.isnan(b) and b > 0 else "blue" for b in bias_monthly]
bars = ax.bar(months_sorted, bias_monthly, color=colors, edgecolor="black", linewidth=1.0)

# x ticks: 1..12 mapped to 3-letter labels, rotated 90 degrees
ax.set_xticks(range(1,13))
ax.set_xticklabels(month_labels, fontsize=TICK_FS, rotation=90)

# y-label (units °C) and font sizes
ax.set_ylabel("Monthly Mean Bias (°C)", fontsize=LABEL_FS, fontweight="bold")
ax.tick_params(axis="y", labelsize=TICK_FS)

# clean spines and grid
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved seasonal bias cycle to: {OUT_PNG}")

In [ ]:
#!/usr/bin/env python3
"""
Modern Seasonal Bias Analysis — Final Fixed Version
==================================================
Fix: Handles 3D masks from shapefiles by collapsing regions 
to produce a single 1D domain-mean time series.
"""

import os
import glob
import re
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import regionmask
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
from scipy import stats

warnings.filterwarnings("ignore")

# ============================================================
# USER CONFIG
# ============================================================
HR_MSWX   = "<SET YOUR PATH HERE>"
HR_DDPM   = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"

OUT_PNG    = "seasonal_bias_modern_masked.png"
DPI        = 300
TEST_START = 2009219
TEST_END   = 2014365

SEASON_MAP = {
    1: ("Winter", "#4A90D9"), 2: ("Winter", "#4A90D9"),
    3: ("Pre-Monsoon", "#E8A838"), 4: ("Pre-Monsoon", "#E8A838"), 5: ("Pre-Monsoon", "#E8A838"),
    6: ("Monsoon", "#2ECC71"), 7: ("Monsoon", "#2ECC71"), 8: ("Monsoon", "#2ECC71"), 9: ("Monsoon", "#2ECC71"),
    10:("Post-Monsoon","#9B59B6"), 11:("Post-Monsoon","#9B59B6"), 12:("Post-Monsoon","#9B59B6"),
}
MONTH_LABELS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# ============================================================
# HELPERS
# ============================================================
def get_token(fname):
    base = os.path.splitext(os.path.basename(fname))[0]
    m = re.search(r"(\d{7,8})", base)
    return int(m.group(1)) if m else None

def token_to_dt(tok):
    s = str(tok)
    return pd.to_datetime(f"{s[:4]}-{s[4:]}", format="%Y-%j") if len(s) == 7 else pd.to_datetime(s, format="%Y%m%d")

def filter_files(files, start, end):
    return [f for f in sorted(files) if (tok := get_token(f)) and start <= tok <= end]

def find_var(ds):
    for v in ("air_temperature","tas","t2m","temperature","temp"):
        if v in ds.data_vars: return v
    return [v for v in ds.data_vars if ds[v].ndim >= 2][0]

def get_india_mask_1d(ds, shp_path):
    """Creates a single boolean mask for all of India."""
    india_gdf = gpd.read_file(shp_path)
    # 3D mask: (region, lat, lon)
    mask_3d = regionmask.mask_3D_geopandas(india_gdf, ds.lon, ds.lat)
    # Collapse 'region' dimension to get a single 2D mask (lat, lon)
    # .any('region') returns True if a pixel belongs to ANY state in the shapefile
    mask_2d = mask_3d.any("region")
    return mask_2d

# ============================================================
# LOAD & PROCESS DATA
# ============================================================
print("Loading files...")
mswx_files = filter_files(glob.glob(os.path.join(HR_MSWX, "*.nc")), TEST_START, TEST_END)
ddpm_files  = filter_files(glob.glob(os.path.join(HR_DDPM,  "*.nc")), TEST_START, TEST_END)

ds_m = xr.open_mfdataset(mswx_files, combine="nested", concat_dim="time")
ds_d = xr.open_mfdataset(ddpm_files,  combine="nested", concat_dim="time")

if "time" not in ds_m.coords:
    ds_m = ds_m.assign_coords(time=[token_to_dt(get_token(f)) for f in mswx_files])
if "time" not in ds_d.coords:
    ds_d = ds_d.assign_coords(time=[token_to_dt(get_token(f)) for f in ddpm_files])

# Create mask
print(f"Generating unified mask from: {os.path.basename(INDIA_SHP)}...")
india_mask = get_india_mask_1d(ds_m, INDIA_SHP)

var_m, var_d = find_var(ds_m), find_var(ds_d)

# Apply mask and compute mean
print("Computing masked spatial means...")
# We mask, then average over spatial dimensions to get (time,) 1D array
mswx_vals = ds_m[var_m].where(india_mask).mean(dim=["lat", "lon"]).values
ddpm_vals = ds_d[var_d].where(india_mask).mean(dim=["lat", "lon"]).values
times     = pd.DatetimeIndex(ds_m["time"].values)

# Kelvin to Celsius
if np.nanmean(mswx_vals) > 200: mswx_vals -= 273.15
if np.nanmean(ddpm_vals) > 200: ddpm_vals -= 273.15

# Final 1D conversion check
mswx_vals = mswx_vals.flatten()
ddpm_vals = ddpm_vals.flatten()

print(f"Confirmed 1D Shapes: Times={len(times)}, MSWX={mswx_vals.shape}, DDPM={ddpm_vals.shape}")

df = pd.DataFrame({
    "date": times, 
    "mswx": mswx_vals, 
    "ddpm": ddpm_vals,
    "bias": ddpm_vals - mswx_vals, 
    "month": times.month, 
    "year": times.year
}).dropna()

# ============================================================
# VISUALIZATION
# ============================================================
fig = plt.figure(figsize=(18, 14), facecolor="white")
gs = gridspec.GridSpec(2, 2, figure=fig, left=0.07, right=0.97, top=0.92, bottom=0.08, wspace=0.3, hspace=0.35)

ax_violin  = fig.add_subplot(gs[0, :])
ax_kde     = fig.add_subplot(gs[1, 0])
ax_heatmap = fig.add_subplot(gs[1, 1])

plt.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})

# Panel A: Violin
for m in range(1, 13):
    sub = df[df["month"] == m]["bias"].values
    color = SEASON_MAP[m][1]
    vp = ax_violin.violinplot(sub, positions=[m], widths=0.7, showextrema=False)
    for pc in vp["bodies"]: pc.set_facecolor(color); pc.set_alpha(0.4)
    q25, q50, q75 = np.percentile(sub, [25, 50, 75])
    ax_violin.vlines(m, sub.min(), sub.max(), color=color, lw=1, alpha=0.3)
    ax_violin.fill_between([m-0.1, m+0.1], q25, q75, color=color, alpha=0.8)
    ax_violin.hlines(q50, m-0.15, m+0.15, color="white", lw=2)

ax_violin.axhline(0, color="black", lw=0.8, ls="--", alpha=0.4)
ax_violin.set_xticks(range(1,13))
ax_violin.set_xticklabels(MONTH_LABELS, fontweight="bold")
ax_violin.set_ylabel("Daily Bias (°C)", fontweight="bold", fontsize=12)
ax_violin.set_title("(a) Monthly Distribution of Masked Daily Bias", loc="left", fontweight="bold", fontsize=14)

# Panel B: KDE
x, y = df["mswx"].values, df["bias"].values
xi, yi = np.mgrid[x.min()-1:x.max()+1:100j, y.min()-0.5:y.max()+0.5:100j]
zi = gaussian_kde(np.vstack([x, y]))(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)
cf = ax_kde.contourf(xi, yi, zi, levels=15, cmap="YlOrRd", alpha=0.8)

for m in range(1, 13):
    sub = df[df["month"] == m]
    ax_kde.scatter(sub["mswx"].mean(), sub["bias"].mean(), s=120, color=SEASON_MAP[m][1], edgecolors="k", zorder=5)

slope, intercept, r, p, _ = stats.linregress(x, y)
ax_kde.plot(x, slope*x + intercept, "k--", lw=1.5, label=f"R={r:.2f}")
ax_kde.set_xlabel("MSWX Temperature (°C)", fontweight="bold")
ax_kde.set_ylabel("Daily Bias (°C)", fontweight="bold")
ax_kde.set_title("(b) Bias–Temperature Phase Portrait", loc="left", fontweight="bold", fontsize=14)
ax_kde.legend()

# Panel C: Heatmap
pivot = df.groupby(["year","month"])["bias"].mean().unstack()
vmax = max(abs(pivot.values.min()), abs(pivot.values.max()))
im = ax_heatmap.imshow(pivot.values, aspect="auto", cmap="RdBu_r", norm=mcolors.TwoSlopeNorm(0, vmin=-vmax, vmax=vmax))

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.iloc[i, j]
        if not np.isnan(val):
            ax_heatmap.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, 
                            color="white" if abs(val) > vmax*0.7 else "black")

ax_heatmap.set_yticks(range(len(pivot.index))); ax_heatmap.set_yticklabels(pivot.index.astype(int))
ax_heatmap.set_xticks(range(12)); ax_heatmap.set_xticklabels(MONTH_LABELS, fontweight="bold")
ax_heatmap.set_title("(c) Year × Month Mean Bias Consistency", loc="left", fontweight="bold", fontsize=14)
plt.colorbar(im, ax=ax_heatmap, label="Mean Bias (°C)")

fig.suptitle("DDPM Performance Analysis: India-Masked Seasonal Bias", fontsize=18, fontweight="bold")
# plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.show()

print(f"Success! Figure saved to {OUT_PNG}")

In [ ]:
#!/usr/bin/env python3
"""
Modern Seasonal Bias Analysis — Section 3.6
============================================
Replaces the monthly mean bias bar chart with a two-panel figure:

  Panel (a): Latitude–Month Hovmöller heatmap of bias (India-masked, zonal mean)
             → reveals WHERE in India the cold bias originates and its seasonal evolution

  Panel (b): Domain-mean seasonal cycle (line) with ±1σ spatial spread (shading)
             + seasonal IMD regime blocks for visual anchoring
             → replaces bar chart with a spread-aware summary

Both panels apply the India political boundary shapefile mask.
All computations are pixel-wise before spatial aggregation.

Run:
    python seasonal_bias_modern.py
"""

import os, glob, re
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import rasterio.features
import rasterio.transform
from shapely.geometry import mapping
import warnings
warnings.filterwarnings("ignore")

# ── USER CONFIG ──────────────────────────────────────────────────────────────
HR_MSWX   = "<SET YOUR PATH HERE>"
HR_DDPM   = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"

TEST_START = 2009219   # YYYYDOY  2009-Aug-07
TEST_END   = 2014365   # YYYYDOY  2014-Dec-31

OUT_PNG = "seasonal_bias_modern.png"
DPI     = 600

LAT_NAMES = ("lat", "latitude", "y")
LON_NAMES = ("lon", "longitude", "x")
VAR_NAMES = ("air_temperature", "tas", "t2m", "temperature", "temp")

# ── HELPERS ──────────────────────────────────────────────────────────────────
def parse_token(fname):
    base = os.path.splitext(os.path.basename(fname))[0]
    m = re.search(r"(\d{7})", base)
    if m: return int(m.group(1))
    m = re.search(r"(\d{8})", base)
    if m: return int(m.group(1))
    return None

def token_to_dt(tok):
    s = str(tok)
    if len(s) == 7:
        return pd.to_datetime(f"{s[:4]}-{s[4:]}", format="%Y-%j")
    return pd.to_datetime(s, format="%Y%m%d")

def filter_files(files, start, end):
    return [f for f in sorted(files)
            if (tok := parse_token(f)) is not None and start <= tok <= end]

def find_var(ds):
    for n in VAR_NAMES:
        if n in ds.data_vars: return n
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating) and ds[v].ndim >= 2:
            return v
    raise RuntimeError("No temperature variable found.")

def find_coord(ds, candidates):
    for n in candidates:
        if n in ds.coords: return n
    raise RuntimeError(f"None of {candidates} in coords: {list(ds.coords)}")

def build_india_mask(shp_path, lat, lon):
    gdf = gpd.read_file(shp_path)
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    geom = gdf.dissolve().geometry.values[0].buffer(0.05)
    nrows, ncols = len(lat), len(lon)
    lon_res = abs(lon[1]-lon[0]) if len(lon)>1 else 0.1
    lat_res = abs(lat[1]-lat[0]) if len(lat)>1 else 0.1
    tfm = rasterio.transform.from_origin(
        lon.min()-lon_res/2, lat.max()+lat_res/2, lon_res, lat_res)
    outside = rasterio.features.geometry_mask(
        [mapping(geom)], out_shape=(nrows,ncols),
        transform=tfm, invert=False, all_touched=True)
    mask = ~outside
    if lat[0] < lat[-1]:      # ascending → flip so row-0 = north
        mask = mask[::-1, :]
    return mask

def maybe_kelvin(arr):
    return arr - 273.15 if np.nanmean(arr) > 200 else arr

# ── LOAD & FILTER FILES ──────────────────────────────────────────────────────
mswx_files = filter_files(glob.glob(os.path.join(HR_MSWX,"*.nc")), TEST_START, TEST_END)
ddpm_files  = filter_files(glob.glob(os.path.join(HR_DDPM, "*.nc")), TEST_START, TEST_END)

if not mswx_files or not ddpm_files:
    raise FileNotFoundError("No files after filtering. Check paths and TEST_START/END.")
print(f"Files: MSWX={len(mswx_files)}, DDPM={len(ddpm_files)}")

mswx_times = [token_to_dt(parse_token(f)) for f in mswx_files]
ddpm_times  = [token_to_dt(parse_token(f)) for f in ddpm_files]

ds_m = (xr.open_mfdataset(mswx_files, combine="nested", concat_dim="time",
                           engine="netcdf4", parallel=False)
          .assign_coords(time=("time", pd.to_datetime(mswx_times)))
          .sortby("time"))
ds_d = (xr.open_mfdataset(ddpm_files,  combine="nested", concat_dim="time",
                           engine="netcdf4", parallel=False)
          .assign_coords(time=("time", pd.to_datetime(ddpm_times)))
          .sortby("time"))

var_m = find_var(ds_m); var_d = find_var(ds_d)
lat_n = find_coord(ds_m, LAT_NAMES)
lon_n = find_coord(ds_m, LON_NAMES)
lat = ds_m[lat_n].values.astype(float)
lon = ds_m[lon_n].values.astype(float)

# ── INDIA MASK ───────────────────────────────────────────────────────────────
print("Building India mask …")
mask_2d = build_india_mask(INDIA_SHP, lat, lon)
print(f"  India pixels: {mask_2d.sum():,}")

# ── LOAD ARRAYS & ALIGN TIMES ────────────────────────────────────────────────
print("Loading arrays …")
arr_m = maybe_kelvin(ds_m[var_m].values)
arr_d = maybe_kelvin(ds_d[var_d].values)

times_m = pd.DatetimeIndex(ds_m.time.values)
times_d = pd.DatetimeIndex(ds_d.time.values)
common  = times_m.intersection(times_d)
arr_m   = arr_m[times_m.get_indexer(common)]
arr_d   = arr_d[times_d.get_indexer(common)]
dates   = common
print(f"  Common dates: {dates[0].date()} → {dates[-1].date()} ({len(dates)} days)")

# pixel-wise bias, masked
bias_3d = arr_d - arr_m                  # (T, nlat, nlon)
bias_3d[:, ~mask_2d] = np.nan

# ── MONTHLY CLIMATOLOGY ──────────────────────────────────────────────────────
month_ids    = np.arange(1, 13)
month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]
months = dates.month

# domain-mean and spatial std per day
bias_flat  = bias_3d.reshape(len(dates), -1)
daily_mean = np.nanmean(bias_flat, axis=1)
daily_std  = np.nanstd(bias_flat,  axis=1)

clim_mean = np.array([daily_mean[months==m].mean() for m in month_ids])
clim_std  = np.array([daily_std [months==m].mean() for m in month_ids])

# ── HOVMÖLLER: zonal-mean bias per (latitude band, month) ───────────────────
# average over longitude pixels only, then over days within each month
hovmoller = np.full((len(lat), 12), np.nan)
for im, m in enumerate(month_ids):
    t_idx = np.where(months==m)[0]
    if len(t_idx) == 0: continue
    # (T_m, nlat, nlon) → mean over time & lon
    hovmoller[:, im] = np.nanmean(bias_3d[t_idx], axis=(0, 2))

# flip so north is at top of heatmap
if lat[0] < lat[-1]:
    hovmoller = hovmoller[::-1, :]
    lat_plot  = lat[::-1]
else:
    lat_plot  = lat

# clip to India lat extent
lat_idx = np.where((lat_plot >= 6.0) & (lat_plot <= 38.0))[0]
hov_sub = hovmoller[lat_idx, :]
lat_sub = lat_plot[lat_idx]

# ── Smooth along latitude axis to suppress single-row NaN speckle ───────────
# 3-pixel Gaussian (sigma=1) preserves real meridional gradients but removes
# artefact striping from sparse zonal averaging at narrow India latitudes.
from scipy.ndimage import gaussian_filter1d
hov_sub_smooth = np.full_like(hov_sub, np.nan)
for col in range(hov_sub.shape[1]):
    col_data = hov_sub[:, col]
    valid = ~np.isnan(col_data)
    if valid.sum() > 5:
        filled = np.where(valid, col_data, np.nanmean(col_data))
        smoothed = gaussian_filter1d(filled, sigma=1.0)
        smoothed[~valid] = np.nan
        hov_sub_smooth[:, col] = smoothed
hov_sub = hov_sub_smooth

# ── FIGURE ───────────────────────────────────────────────────────────────────
IMD_SEASONS = {
    "Winter (JF)":         ([1,2],     "#4e9af1"),
    "Pre-Monsoon (MAM)":   ([3,4,5],   "#f4a261"),
    "Monsoon (JJAS)":      ([6,7,8,9], "#2a9d8f"),
    "Post-Monsoon (OND)":  ([10,11,12],"#e76f51"),
}

fig = plt.figure(figsize=(16, 6.5))
fig.patch.set_facecolor("#fafafa")
gs  = GridSpec(1, 2, figure=fig, width_ratios=[1, 1.3], wspace=0.38)

# ── Panel (a): Hovmöller ─────────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
ax_a.set_facecolor("#f0f0f0")

vmax = np.nanpercentile(np.abs(hov_sub), 98)
# shading='nearest' uses cell-centre coords matching C shape exactly.
# X = month centres 1..12, Y = latitude centres → C must be (nY, nX) = (320, 12) ✓
im_h = ax_a.pcolormesh(
    np.arange(1, 13), lat_sub, hov_sub,
    cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='nearest')

cbar = fig.colorbar(im_h, ax=ax_a, fraction=0.04, pad=0.03)
cbar.set_label("Zonal-mean bias (°C)", fontsize=10, fontweight='bold')
cbar.ax.tick_params(labelsize=9)

# season dividers
for sb in [2.5, 5.5, 9.5]:
    ax_a.axvline(sb, color='white', lw=1.4, ls='--', alpha=0.75)

ax_a.set_xticks(np.arange(1,13))
ax_a.set_xticklabels(month_labels, fontsize=10)
ax_a.set_ylabel("Latitude (°N)", fontsize=12, fontweight='bold')
ax_a.set_xlabel("Month", fontsize=12, fontweight='bold')
ax_a.tick_params(labelsize=10)
ax_a.yaxis.set_major_locator(mticker.MultipleLocator(5))
ax_a.set_ylim(lat_sub.min(), lat_sub.max())
ax_a.text(-0.18, 1.04, "(a)", transform=ax_a.transAxes,
          fontsize=13, fontweight='bold', va='top')

# ── Panel (b): domain-mean cycle + spread ────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
ax_b.set_facecolor("#f7f7f7")
x = np.arange(1, 13)

# season background blocks
ypad = clim_mean.min() - clim_std.max()*1.3
for label, (ms, color) in IMD_SEASONS.items():
    xmin = ms[0]-0.5; xmax = ms[-1]+0.5
    ax_b.axvspan(xmin, xmax, color=color, alpha=0.10, zorder=0)
    ax_b.text((xmin+xmax)/2, ypad,
              label, ha='center', va='bottom', fontsize=8.5,
              color=color, fontweight='bold', alpha=0.9)

ax_b.axhline(0, color='#444444', lw=0.9, ls='--', alpha=0.5, zorder=1)

# ±1σ shading
ax_b.fill_between(x, clim_mean-clim_std, clim_mean+clim_std,
                  color='#2563eb', alpha=0.15,
                  label='±1σ spatial spread (India pixels)', zorder=2)

# domain-mean line
ax_b.plot(x, clim_mean, color='#1d4ed8', lw=2.2, marker='o',
          markersize=6, markerfacecolor='white',
          markeredgecolor='#1d4ed8', markeredgewidth=1.8,
          label='India domain-mean bias', zorder=3)

# annotate min and max bias months
imin = np.argmin(clim_mean)
imax = np.argmax(clim_mean)
for ann_xi, ann_y, dy, label_sfx in [
        (imin, clim_mean[imin], -1, "peak cold"),
        (imax, clim_mean[imax], +1, "peak warm")]:
    ax_b.annotate(
        f"{clim_mean[ann_xi]:+.3f}°C\n({month_labels[ann_xi]})",
        xy=(x[ann_xi], clim_mean[ann_xi]),
        xytext=(x[ann_xi] + 1.0,
                clim_mean[ann_xi] + dy * max(clim_std[ann_xi], 0.05)),
        fontsize=9, color='#1d4ed8', fontweight='bold',
        ha='left',
        arrowprops=dict(arrowstyle='->', color='#1d4ed8', lw=1.2))

ax_b.set_xticks(x)
ax_b.set_xticklabels(month_labels, fontsize=11)
ax_b.set_ylabel("Bias  (DDPM − MSWX, °C)", fontsize=12, fontweight='bold')
# ax_b.set_xlabel("Month", fontsize=12, fontweight='bold')
ax_b.tick_params(labelsize=11)
ax_b.legend(fontsize=10, loc='lower left', framealpha=0.9)
ax_b.spines[['top','right']].set_visible(False)
ax_b.set_xlim(0.5, 12.5)
# keep y-axis tightly around the actual data range
ymargin = max(clim_std) * 1.5
ax_b.set_ylim(clim_mean.min() - ymargin, clim_mean.max() + ymargin)
ax_b.text(-0.12, 1.04, "(b)", transform=ax_b.transAxes,
          fontsize=13, fontweight='bold', va='top')

fig.suptitle(
    "Seasonal Bias Characteristics of DDPM Downscaling  (India political boundary mask, 2009–2014)",
    fontsize=12, fontweight='bold', y=1.02)

# plt.savefig(OUT_PNG, dpi=DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
# print(f"\nSaved → {OUT_PNG}")

# ── MANUSCRIPT STATS ─────────────────────────────────────────────────────────
print("\n── Monthly Statistics (India-masked) ──")
print(f"{'Month':<6} {'Mean':>10} {'Spatial ±1σ':>13}")
print("-"*32)
for im, m in enumerate(month_ids):
    print(f"{month_labels[im]:<6} {clim_mean[im]:>+9.4f}°C  "
          f"±{clim_std[im]:.4f}°C")

print(f"\nPeak cold bias : {month_labels[np.argmin(clim_mean)]} "
      f"({clim_mean.min():+.4f}°C)")
print(f"Minimum bias   : {month_labels[np.argmax(clim_mean)]} "
      f"({clim_mean.max():+.4f}°C)")
print(f"Annual range   : {clim_mean.max()-clim_mean.min():.4f}°C")
print()
for label, (ms, _) in IMD_SEASONS.items():
    idx = [m-1 for m in ms]
    smean = clim_mean[idx].mean()
    sstd  = clim_std[idx].mean()
    print(f"{label:<25}: {smean:+.4f}°C (mean spatial spread ±{sstd:.4f}°C)")

In [ ]:
#!/usr/bin/env python3
"""
Seasonal Bias Spatial Maps — Section 3.6 (India Focused)
================================================================
Four-panel figure showing spatial distribution of mean bias (DDPM − MSWX, °C)
across India's four IMD meteorological seasons.
"""

import os, glob, re
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import regionmask
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
from matplotlib.colors import TwoSlopeNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import warnings

warnings.filterwarnings("ignore")

# ── USER CONFIG ──────────────────────────────────────────────────────────────
HR_MSWX     = "<SET YOUR PATH HERE>"
HR_DDPM     = "<SET YOUR PATH HERE>"
INDIA_SHP   = "<SET YOUR PATH HERE>"
SUBDIV_SHP  = None  # e.g. "/path/to/imd_subdivisions.shp"

TEST_START  = 2009219
TEST_END    = 2014365

OUT_PNG = "seasonal_bias_maps_india_focused.png"
DPI     = 300

# IMD seasons: label, months, panel letter
SEASONS = [
    ("Winter (JF)",           [1, 2],         "(a)"),
    ("Pre-Monsoon (MAM)",    [3, 4, 5],      "(b)"),
    ("Monsoon (JJAS)",       [6, 7, 8, 9],   "(c)"),
    ("Post-Monsoon (OND)",   [10, 11, 12],   "(d)"),
]

# ── HELPERS ──────────────────────────────────────────────────────────────────
def parse_token(fname):
    base = os.path.splitext(os.path.basename(fname))[0]
    m = re.search(r"(\d{7,8})", base)
    return int(m.group(1)) if m else None

def token_to_dt(tok):
    s = str(tok)
    return pd.to_datetime(f"{s[:4]}-{s[4:]}", format="%Y-%j") if len(s) == 7 else pd.to_datetime(s, format="%Y%m%d")

def filter_files(files, start, end):
    return [f for f in sorted(files) if (tok := parse_token(f)) is not None and start <= tok <= end]

def find_var(ds):
    for n in ("air_temperature", "tas", "t2m", "temperature", "temp"):
        if n in ds.data_vars: return n
    return [v for v in ds.data_vars if ds[v].ndim >= 2][0]

def find_coord(ds, candidates):
    for n in candidates:
        if n in ds.coords: return n
    return None

# ── LOAD & FILTER ────────────────────────────────────────────────────────────
mswx_files = filter_files(glob.glob(os.path.join(HR_MSWX, "*.nc")), TEST_START, TEST_END)
ddpm_files  = filter_files(glob.glob(os.path.join(HR_DDPM,  "*.nc")), TEST_START, TEST_END)
print(f"Files: MSWX={len(mswx_files)}, DDPM={len(ddpm_files)}")

ds_m = (xr.open_mfdataset(mswx_files, combine="nested", concat_dim="time")
          .assign_coords(time=("time", [token_to_dt(parse_token(f)) for f in mswx_files]))
          .sortby("time"))
ds_d = (xr.open_mfdataset(ddpm_files, combine="nested", concat_dim="time")
          .assign_coords(time=("time", [token_to_dt(parse_token(f)) for f in ddpm_files]))
          .sortby("time"))

var_m, var_d = find_var(ds_m), find_var(ds_d)
lat_n = find_coord(ds_m, ("lat", "latitude", "y"))
lon_n = find_coord(ds_m, ("lon", "longitude", "x"))
lat, lon = ds_m[lat_n].values, ds_m[lon_n].values

# ── INDIA MASKING (RegionMask) ────────────────────────────────────────────────
print(f"Applying India Boundary Mask from: {os.path.basename(INDIA_SHP)}...")
india_gdf = gpd.read_file(INDIA_SHP)
mask = regionmask.mask_3D_geopandas(india_gdf, ds_m.lon, ds_m.lat).any("region")

# ── LOAD ARRAYS & CALCULATE BIAS ──────────────────────────────────────────────
print("Processing seasonal means...")
def clean_arr(ds, var):
    arr = ds[var].where(mask).values
    return arr - 273.15 if np.nanmean(arr) > 200 else arr

arr_m = clean_arr(ds_m, var_m)
arr_d = clean_arr(ds_d, var_d)
bias_3d = arr_d - arr_m
times = pd.DatetimeIndex(ds_m.time.values)

# ── SEASONAL MEAN BIAS MAPS ──────────────────────────────────────────────────
season_maps, season_stats = {}, {}
for name, ms, _ in SEASONS:
    t_idx = np.where(np.isin(times.month, ms))[0]
    smap = np.nanmean(bias_3d[t_idx], axis=0)
    india_vals = smap[~np.isnan(smap)]
    season_maps[name] = smap
    season_stats[name] = {
        "mean": float(np.nanmean(india_vals)),
        "p05": float(np.nanpercentile(india_vals, 5)),
        "p95": float(np.nanpercentile(india_vals, 95)),
        "n_days": len(t_idx)
    }

# Colorbar limits (98th percentile of abs bias)
vmax = max(0.1, float(np.nanpercentile(np.abs(bias_3d[~np.isnan(bias_3d)]), 98)))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
cmap = plt.cm.RdBu_r

# ── FIGURE ───────────────────────────────────────────────────────────────────
PROJ = ccrs.PlateCarree()
EXTENT = [67, 98, 6, 38]  # Focused on India region

fig = plt.figure(figsize=(15, 12), facecolor="white")
from matplotlib.gridspec import GridSpec
gs = GridSpec(2, 3, figure=fig, width_ratios=[1, 1, 0.05], hspace=0.1, wspace=0.1)

axes = [fig.add_subplot(gs[0, 0], projection=PROJ), fig.add_subplot(gs[0, 1], projection=PROJ),
        fig.add_subplot(gs[1, 0], projection=PROJ), fig.add_subplot(gs[1, 1], projection=PROJ)]
cbar_ax = fig.add_subplot(gs[:, 2])

# Overlay prep
india_boundary = india_gdf.to_crs(epsg=4326)

for ax, (name, ms, letter) in zip(axes, SEASONS):
    ax.set_extent(EXTENT, crs=PROJ)
    
    pcm = ax.pcolormesh(lon, lat, season_maps[name], cmap=cmap, norm=norm, transform=PROJ, shading='auto', rasterized=True)
    india_boundary.boundary.plot(ax=ax, color='#111111', linewidth=1.0, transform=PROJ)
    
    # Gridlines
    gl = ax.gridlines(crs=PROJ, draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = gl.right_labels = False
    gl.left_labels = letter in ["(a)", "(c)"]; gl.bottom_labels = letter in ["(c)", "(d)"]
    gl.xlabel_style = gl.ylabel_style = {'size': 9}
    gl.xlocator = mticker.FixedLocator([70, 80, 90])
    gl.ylocator = mticker.FixedLocator([10, 20, 30])

    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.text(0.02, 0.98, letter, transform=ax.transAxes, fontsize=14, fontweight='bold', va='top', ha='left',
            path_effects=[pe.withStroke(linewidth=3, foreground='white')])
    
    # Stats Annotation
    s = season_stats[name]
    ax.text(0.98, 0.02, f"Mean: {s['mean']:+.3f}°C\nP05: {s['p05']:+.3f}°C\nP95: {s['p95']:+.3f}°C",
            transform=ax.transAxes, fontsize=12, va='bottom', ha='right', family='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

# Colorbar
cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax, extend='both')
cb.set_label("Mean Bias (DDPM − MSWX, °C)", fontsize=12, fontweight='bold')
cb.ax.axhline(0, color='black', lw=1.5)

#fig.suptitle("DDPM Performance: Seasonal Spatial Bias (India Focused)", fontsize=20, fontweight='bold', y=0.98)
plt.savefig(OUT_PNG, dpi=DPI, bbox_inches='tight')
plt.show()

print(f"\nSaved → {OUT_PNG}")

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

# -----------------------------
# Paths
# -----------------------------
hr_mswx = "<SET YOUR PATH HERE>"
hr_ddpm = "<SET YOUR PATH HERE>"

# -----------------------------
# Helper functions
# -----------------------------
def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2003079, end=2014365):
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

# -----------------------------
# Step 1: Load files
# -----------------------------
f_mswx = sorted(glob.glob(os.path.join(hr_mswx, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(hr_ddpm, "*_ddpm.nc")))

f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"Found {len(f_mswx)} MSWX files and {len(f_ddpm)} DDPM files")

# -----------------------------
# Step 2: Open datasets
# -----------------------------
ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# -----------------------------
# Step 3: Assign time and drop timezone
# -----------------------------
if "time" not in ds_mswx:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

ds_mswx["time"] = pd.to_datetime(ds_mswx["time"].values)
ds_ddpm["time"] = pd.to_datetime(ds_ddpm["time"].values)

# -----------------------------
# Step 4: Compute monthly climatology
# -----------------------------
# Use `groupby('time.month')` to average across all years for each month
monthly_climatology_mswx = ds_mswx["air_temperature"].mean(dim=["lat", "lon"]).groupby('time.month').mean('time')
monthly_climatology_ddpm = ds_ddpm["air_temperature"].mean(dim=["lat", "lon"]).groupby('time.month').mean('time')

# Calculate the climatological bias
climatological_bias = monthly_climatology_ddpm - monthly_climatology_mswx

# -----------------------------
# Step 5: Plotting
# -----------------------------
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

months = range(1, 13)
month_labels = [pd.to_datetime(str(m), format='%m').strftime('%b') for m in months]

# Plot 1: Climatological Mean Temperatures
ax1.plot(months, monthly_climatology_mswx, label="MSWX Climatology", marker='o', color="blue", linewidth=2)
ax1.plot(months, monthly_climatology_ddpm, label="DDPM Climatology", marker='s', color="red", linestyle='--', linewidth=2)
ax1.set_title("Long-Term Monthly Temperature Climatology (2003-2014)", fontsize=16)
ax1.set_ylabel("Temperature (°C)", fontsize=12)
ax1.set_xticks(months)
ax1.set_xticklabels(month_labels)
ax1.legend()
ax1.grid(True)

# Plot 2: Climatological Bias
ax2.plot(months, climatological_bias, label="Climatological Bias (DDPM - MSWX)", marker='o', color="purple", linewidth=2)
ax2.axhline(0, color='black', linestyle='--', linewidth=1)
ax2.set_title("Long-Term Monthly Bias Climatology", fontsize=16)
ax2.set_xlabel("Month", fontsize=12)
ax2.set_ylabel("Bias (°C)", fontsize=12)
ax2.set_xticks(months)
ax2.set_xticklabels(month_labels)
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
#!/usr/bin/env python3
"""
RMSE Spatial Distribution Analysis - IMD Climatic Zones
Q1 Journal Quality with Official India Meteorological Department Zones

Comprehensive spatial error analysis including:
- 36 official IMD climatic zones stratification
- Zone-based statistics with comprehensive reporting
- Baseline comparison (climatology)
- Uncertainty quantification
- Statistical significance testing
- Professional publication-quality output

Author: Climate Downscaling Analysis
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.colors as colors
import glob
import os
import rioxarray
import geopandas as gpd
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION
# =====================================================================

HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"
IMD_ZONES_SHP = "<SET YOUR PATH HERE>"
OUTPUT_DIR = "<SET YOUR PATH HERE>"

os.makedirs(OUTPUT_DIR, exist_ok=True)

PLOT_MAIN = "rmse_spatial_analysis_main_imd.pdf"
PLOT_ZONES_MAP = "rmse_imd_zones_map.pdf"
PLOT_ZONES_STATS = "rmse_imd_zones_statistics.pdf"
STATS_FILE = "rmse_analysis_statistics_imd.txt"
ZONES_CSV = "rmse_imd_zones_statistics.csv"

# =====================================================================
# HELPER FUNCTIONS
# =====================================================================

def get_file_date(fname):
    """Extract YYYYDOY from filename."""
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)


def doy_to_datetime(yydoy):
    """Convert YYYYDOY to datetime."""
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")


def filter_period(files, start=2009219, end=2014365):
    """Filter files within date range."""
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid


def compute_bilinear_rmse(ds_mswx, ds_ddpm):
    """Compute RMSE using climatology as baseline."""
    clim = ds_mswx["air_temperature"].mean(dim="time")
    squared_error_clim = (clim - ds_mswx["air_temperature"])**2
    rmse_clim = np.sqrt(squared_error_clim.mean(dim="time"))
    return rmse_clim


# =====================================================================
# MAIN EXECUTION
# =====================================================================

print("="*80)
print("RMSE SPATIAL ANALYSIS - IMD CLIMATIC ZONES (36 ZONES)")
print("="*80)
print()

# -------------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------------
print("Step 1: Loading data...")

f_mswx = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))

f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"  Found {len(f_mswx)} MSWX files")
print(f"  Found {len(f_ddpm)} DDPM files")

ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# -------------------------------------------------------
# 2. ASSIGN TIME
# -------------------------------------------------------
print("\nStep 2: Assigning time coordinates...")

if "time" not in ds_mswx.coords:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

ds_mswx["time"] = pd.to_datetime(ds_mswx["time"].values)
ds_ddpm["time"] = pd.to_datetime(ds_ddpm["time"].values)

# -------------------------------------------------------
# 3. COMPUTE RMSE
# -------------------------------------------------------
print("\nStep 3: Computing RMSE...")

squared_error = (ds_ddpm["air_temperature"] - ds_mswx["air_temperature"])**2
mean_squared_error = squared_error.mean(dim="time")
rmse = np.sqrt(mean_squared_error)

print(f"  RMSE range: {rmse.min().values:.3f} - {rmse.max().values:.3f} °C")

# -------------------------------------------------------
# 4. COMPUTE BASELINE RMSE
# -------------------------------------------------------
print("Step 4: Computing baseline RMSE (climatology)...")

rmse_baseline = compute_bilinear_rmse(ds_mswx, ds_ddpm)

# -------------------------------------------------------
# 5. PREPARE FOR SPATIAL ANALYSIS
# -------------------------------------------------------
print("\nStep 5: Preparing spatial data...")

rmse_renamed = rmse.rename({"lat": "y", "lon": "x"})
rmse_renamed.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
rmse_renamed.rio.write_crs("EPSG:4326", inplace=True)

rmse_baseline_renamed = rmse_baseline.rename({"lat": "y", "lon": "x"})
rmse_baseline_renamed.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
rmse_baseline_renamed.rio.write_crs("EPSG:4326", inplace=True)

# Load shapefiles
gdf_india = gpd.read_file(INDIA_SHP)
rmse_clipped = rmse_renamed.rio.clip(gdf_india.geometry, gdf_india.crs)
rmse_baseline_clipped = rmse_baseline_renamed.rio.clip(gdf_india.geometry, gdf_india.crs)

# Load IMD zones
print("\nStep 5b: Loading IMD climatic zones...")
gdf_zones = gpd.read_file(IMD_ZONES_SHP)
print(f"  Found {len(gdf_zones)} zones")
print(f"  Zones: {gdf_zones['ST_NM'].unique()}")

# Extract lat/lon
lat = rmse_clipped.y.values
lon = rmse_clipped.x.values

print(f"  Domain: {len(lat)} × {len(lon)} grid points")

# -------------------------------------------------------
# 6. COMPUTE GLOBAL STATISTICS
# -------------------------------------------------------
print("\nStep 6: Computing global spatial statistics...")

rmse_values = rmse_clipped.values
rmse_values_valid = rmse_values[~np.isnan(rmse_values)]

global_stats = {
    'mean': float(np.nanmean(rmse_values)),
    'median': float(np.nanmedian(rmse_values)),
    'std': float(np.nanstd(rmse_values)),
    'min': float(np.nanmin(rmse_values)),
    'max': float(np.nanmax(rmse_values)),
    'p25': float(np.nanpercentile(rmse_values, 25)),
    'p75': float(np.nanpercentile(rmse_values, 75)),
    'p95': float(np.nanpercentile(rmse_values, 95)),
    'n_valid': int(np.sum(~np.isnan(rmse_values))),
}

baseline_values = rmse_baseline_clipped.values
baseline_stats = {
    'mean': float(np.nanmean(baseline_values)),
    'median': float(np.nanmedian(baseline_values)),
    'std': float(np.nanstd(baseline_values)),
}

improvement = ((baseline_stats['mean'] - global_stats['mean']) / 
               baseline_stats['mean'] * 100)

print(f"\nGlobal RMSE Statistics:")
print(f"  Mean:        {global_stats['mean']:.4f} °C")
print(f"  Median:      {global_stats['median']:.4f} °C")
print(f"  Std Dev:     {global_stats['std']:.4f} °C")
print(f"  95th %ile:   {global_stats['p95']:.4f} °C")

print(f"\nBaseline Comparison:")
print(f"  Climatology RMSE: {baseline_stats['mean']:.4f} °C")
print(f"  Improvement:      {improvement:.1f}%")

# -------------------------------------------------------
# 7. IMD ZONES ANALYSIS
# -------------------------------------------------------
print("\nStep 7: Computing statistics for 36 IMD zones...")

zone_stats = {}

# We don't need manual numpy meshing. We will use rioxarray's clip function 
# which is optimized for this exact task.

for idx, zone_row in gdf_zones.iterrows():
    zone_name = zone_row['ST_NM']
    
    try:
        # Use rioxarray to clip the data to the specific zone geometry
        # drop=True shrinks the array to the bounding box of the zone (faster)
        # all_touched=True ensures we catch grid cells at the edge of irregular zones
        zone_subset = rmse_clipped.rio.clip(
            [zone_row.geometry], 
            gdf_zones.crs, 
            drop=True, 
            all_touched=True
        )
        
        # Flatten and remove NaNs (pixels outside the polygon but inside bounding box become NaN)
        zone_rmse = zone_subset.values.flatten()
        zone_rmse_valid = zone_rmse[~np.isnan(zone_rmse)]
        
        if len(zone_rmse_valid) > 0:
            zone_stats[zone_name] = {
                'mean': float(np.nanmean(zone_rmse_valid)),
                'std': float(np.nanstd(zone_rmse_valid)),
                'min': float(np.nanmin(zone_rmse_valid)),
                'max': float(np.nanmax(zone_rmse_valid)),
                'median': float(np.nanmedian(zone_rmse_valid)),
                'p25': float(np.nanpercentile(zone_rmse_valid, 25)),
                'p75': float(np.nanpercentile(zone_rmse_valid, 75)),
                'n_valid': int(len(zone_rmse_valid)),
                # Check if 'cat' exists, otherwise set a default
                'cat': int(zone_row['cat']) if 'cat' in zone_row else 0,
            }
            
            print(f"  {zone_name:<30}: {zone_stats[zone_name]['mean']:.3f} °C "
                  f"(n={zone_stats[zone_name]['n_valid']:,})")
                  
    except Exception as e:
        # This catches cases where a zone might not overlap with your raster extent
        print(f"  Skipping {zone_name}: {e}")
        continue

print(f"\n✓ Statistics computed for {len(zone_stats)} zones")

# -------------------------------------------------------
# 8. STATISTICAL TESTING
# -------------------------------------------------------
print("\nStep 8: Running statistical tests...")

# Prepare data for Kruskal-Wallis test
zones_data = []
zones_names = []

# We use rioxarray.clip again because it is significantly faster 
# than iterating through pixels with shapely
for zone_name in sorted(zone_stats.keys()):
    try:
        # Get the geometry for this specific zone
        zone_geom = gdf_zones[gdf_zones['ST_NM'] == zone_name].geometry.values
        
        # Clip the RMSE raster to this zone
        # drop=True shrinks the raster to the zone bounds (faster)
        zone_clip = rmse_clipped.rio.clip(
            zone_geom, 
            gdf_zones.crs, 
            drop=True, 
            all_touched=True
        )
        
        # Flatten and remove NaNs to get raw values
        vals = zone_clip.values.flatten()
        vals_clean = vals[~np.isnan(vals)]
        
        if len(vals_clean) > 0:
            zones_data.append(vals_clean)
            zones_names.append(zone_name)
            
    except Exception as e:
        # Handle zones that might be outside the raster extent
        continue

if len(zones_data) > 1:
    # Perform Kruskal-Wallis H-test
    h_stat, p_val = stats.kruskal(*zones_data)
    print(f"\nKruskal-Wallis Test (IMD Zone Differences):")
    print(f"  H-statistic: {h_stat:.4f}")
    print(f"  P-value:     {p_val:.2e}")
    print(f"  Result:      {'Significant' if p_val < 0.05 else 'Not significant'}")
else:
    print("\nInsufficient data for statistical testing.")
    h_stat, p_val = np.nan, np.nan

# -------------------------------------------------------
# 9. SAVE STATISTICS
# -------------------------------------------------------
print("\nStep 9: Saving statistics...")

try:
    with open(os.path.join(OUTPUT_DIR, STATS_FILE), 'w') as f:
        f.write("="*80 + "\n")
        f.write("RMSE SPATIAL ANALYSIS STATISTICS - 36 IMD CLIMATIC ZONES\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Analysis Period: 2009-2014\n")
        f.write(f"Domain: {len(lat)} × {len(lon)} grid points\n")
        f.write(f"Valid grid cells: {global_stats['n_valid']:,}\n")
        f.write(f"Number of zones: {len(zone_stats)}\n\n")
        
        f.write("GLOBAL RMSE STATISTICS\n")
        f.write("-"*80 + "\n")
        for key, val in global_stats.items():
            f.write(f"{key:<15}: {val}\n")
        
        f.write("\n\nBASELINE COMPARISON (Climatology)\n")
        f.write("-"*80 + "\n")
        f.write(f"Climatology RMSE: {baseline_stats['mean']:.4f} °C\n")
        f.write(f"DDPM RMSE:        {global_stats['mean']:.4f} °C\n")
        f.write(f"Improvement:      {improvement:.1f}%\n")
        
        f.write("\n\nIMD ZONE STATISTICS (36 ZONES)\n")
        f.write("-"*80 + "\n")
        f.write(f"{'Zone Name':<30} {'Mean':<10} {'Std':<10} {'Median':<10} {'N Valid':<10}\n")
        f.write("-"*80 + "\n")
        
        for zone_name in sorted(zone_stats.keys()):
            stats_dict = zone_stats[zone_name]
            f.write(f"{zone_name:<30} "
                   f"{stats_dict['mean']:<10.4f} "
                   f"{stats_dict['std']:<10.4f} "
                   f"{stats_dict['median']:<10.4f} "
                   f"{stats_dict['n_valid']:<10}\n")
        
        f.write("\n\nSTATISTICAL TESTS\n")
        f.write("-"*80 + "\n")
        f.write(f"Kruskal-Wallis Test (IMD Zone Differences):\n")
        f.write(f"  H-statistic: {h_stat:.4f}\n")
        f.write(f"  P-value:     {p_val:.2e}\n")
        f.write(f"  Number of zones: {len(zone_stats)}\n")
        f.write(f"  Result:      {'Significant' if p_val < 0.05 else 'Not significant'} (α=0.05)\n")
        
        f.write("\n\nZONE RANKING BY MEAN RMSE\n")
        f.write("-"*80 + "\n")
        sorted_zones = sorted(zone_stats.items(), key=lambda x: x[1]['mean'])
        
        f.write("\nBEST PERFORMING ZONES (Lowest RMSE):\n")
        for i, (zone_name, stats_dict) in enumerate(sorted_zones[:10], 1):
            f.write(f"{i:2d}. {zone_name:<30}: {stats_dict['mean']:.4f} °C\n")
        
        f.write("\nWORST PERFORMING ZONES (Highest RMSE):\n")
        for i, (zone_name, stats_dict) in enumerate(sorted_zones[-10:], 1):
            f.write(f"{i:2d}. {zone_name:<30}: {stats_dict['mean']:.4f} °C\n")
    
    print(f"  ✓ Statistics saved to {STATS_FILE}")
except Exception as e:
    print(f"  ✗ Error saving: {e}")

# Save zones statistics as CSV
try:
    zones_df = pd.DataFrame(zone_stats).T.sort_values('mean')
    zones_df.to_csv(os.path.join(OUTPUT_DIR, ZONES_CSV))
    print(f"  ✓ Zone statistics saved to {ZONES_CSV}")
except Exception as e:
    print(f"  ✗ Error saving zones CSV: {e}")

# =====================================================================
# PLOTTING
# =====================================================================
print("\nStep 10: Creating publication-quality figures...\n")

# -------------------------------------------------------
# FIGURE 1: MAIN 4-PANEL ANALYSIS
# -------------------------------------------------------
print("  Generating main analysis figure...")

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.25)

# Panel 1a: RMSE Map
ax1 = fig.add_subplot(gs[0, 0])
im1 = rmse_clipped.plot(ax=ax1, cmap="YlOrRd", add_colorbar=False, x='x', y='y')
gdf.boundary.plot(ax=ax1, edgecolor='black', linewidth=1)
ax1.set_xlabel('Longitude (°E)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Latitude (°N)', fontsize=11, fontweight='bold')
ax1.set_title('(a) Spatial RMSE Distribution', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
ax1.tick_params(labelsize=10)

# Add statistics box
stats_text = (f"Mean: {global_stats['mean']:.3f} °C\n"
              f"Median: {global_stats['median']:.3f} °C\n"
              f"Std: {global_stats['std']:.3f} °C\n"
              f"95th %ile: {global_stats['p95']:.3f} °C")
ax1.text(0.98, 0.98, stats_text, transform=ax1.transAxes, # X-coordinate changed to 0.98
        fontsize=9, verticalalignment='top', horizontalalignment='right', # Added horizontalalignment
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

# Panel 1b: Histogram
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(rmse_values_valid, bins=50, color='coral', edgecolor='black', alpha=0.7)
ax2.axvline(global_stats['mean'], color='red', linestyle='--', linewidth=2, 
           label=f"Mean: {global_stats['mean']:.3f} °C")
ax2.axvline(global_stats['median'], color='blue', linestyle='--', linewidth=2,
           label=f"Median: {global_stats['median']:.3f} °C")
ax2.set_xlabel('RMSE (°C)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax2.set_title('(b) RMSE Distribution', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')
ax2.tick_params(labelsize=10)

# Panel 1c: Top 15 zones by RMSE (bar chart)
ax3 = fig.add_subplot(gs[1, 0])
top_zones = sorted(zone_stats.items(), key=lambda x: x[1]['mean'])[:15]
zone_names_top = [z[0][:20] for z in top_zones]
zone_means_top = [z[1]['mean'] for z in top_zones]
zone_stds_top = [z[1]['std'] for z in top_zones]

ax3.barh(range(len(zone_names_top)), zone_means_top, xerr=zone_stds_top,
        color='skyblue', edgecolor='black', linewidth=1)
ax3.set_yticks(range(len(zone_names_top)))
ax3.set_yticklabels(zone_names_top, fontsize=8)
ax3.set_xlabel('Mean RMSE (°C)', fontsize=11, fontweight='bold')
ax3.set_title('(c) Best 15 IMD Zones by RMSE', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')
ax3.invert_yaxis()

# Panel 1d: Comparison to Baseline
ax4 = fig.add_subplot(gs[1, 1])
methods = ['Climatology\n(Baseline)', 'DDPM']
means = [baseline_stats['mean'], global_stats['mean']]
colors_bar = ['lightgray', 'lightgreen']
bars = ax4.bar(methods, means, color=colors_bar, edgecolor='black', linewidth=2)
ax4.set_ylabel('RMSE (°C)', fontsize=11, fontweight='bold')
ax4.set_title(f'(d) Comparison to Baseline\n(DDPM {improvement:.1f}% better)', 
             fontsize=12, fontweight='bold')
ax4.tick_params(labelsize=10)
ax4.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, means):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f} °C', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add colorbar
cbar_ax = fig.add_axes([0.22, 0.08, 0.15, 0.01])
cbar = fig.colorbar(im1, cax=cbar_ax, orientation='horizontal')
cbar.set_label('RMSE (°C)', fontsize=10, fontweight='bold')

plt.suptitle('RMSE Spatial Analysis with 36 IMD Climatic Zones: DDPM vs MSWX (2009-2014)', 
            fontsize=14, fontweight='bold', y=0.995)
plt.savefig(os.path.join(OUTPUT_DIR, PLOT_MAIN), dpi=300, bbox_inches='tight')
print(f"  ✓ Saved to {PLOT_MAIN}")

# -------------------------------------------------------
# FIGURE 2: ALL 36 ZONES STATISTICS
# -------------------------------------------------------
print("  Generating comprehensive zones statistics figure...")

fig2 = plt.figure(figsize=(18, 12))
gs2 = GridSpec(2, 2, figure=fig2, hspace=0.4, wspace=0.25)

# Panel 2a: All zones sorted by mean RMSE
ax = fig2.add_subplot(gs2[0, :])
all_zones = sorted(zone_stats.items(), key=lambda x: x[1]['mean'])
zone_names_all = [z[0] for z in all_zones]
zone_means_all = [z[1]['mean'] for z in all_zones]
zone_stds_all = [z[1]['std'] for z in all_zones]

colors_zones = plt.cm.RdYlGn_r(np.linspace(0, 1, len(all_zones)))
bars = ax.bar(range(len(zone_names_all)), zone_means_all, yerr=zone_stds_all,
             color=colors_zones, edgecolor='black', linewidth=0.5,
             capsize=3, error_kw={'elinewidth': 1})

ax.set_xticks(range(len(zone_names_all)))
ax.set_xticklabels(zone_names_all, rotation=90, fontsize=7)
ax.set_ylabel('Mean RMSE (°C)', fontsize=11, fontweight='bold')
ax.set_title('(a) All 36 IMD Zones Ranked by Mean RMSE with Std Dev', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(global_stats['mean'], color='red', linestyle='--', linewidth=2, label=f"Global mean: {global_stats['mean']:.3f} °C")
ax.legend(fontsize=10)

# Panel 2b: Distribution statistics
ax = fig2.add_subplot(gs2[1, 0])
zone_names_sorted = [z[0] for z in all_zones]
zone_stds = [zone_stats[z]['std'] for z in zone_names_sorted]
ax.scatter(zone_means_all, zone_stds, s=100, alpha=0.6, edgecolors='black')
ax.set_xlabel('Mean RMSE (°C)', fontsize=11, fontweight='bold')
ax.set_ylabel('Standard Deviation (°C)', fontsize=11, fontweight='bold')
ax.set_title('(b) Mean vs Std Dev by Zone', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Panel 2c: Sample size
ax = fig2.add_subplot(gs2[1, 1])
zone_n_valid = [zone_stats[z]['n_valid'] for z in zone_names_sorted]
ax.scatter(zone_means_all, zone_n_valid, s=100, alpha=0.6, edgecolors='black')
ax.set_xlabel('Mean RMSE (°C)', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Valid Grid Cells', fontsize=11, fontweight='bold')
ax.set_title('(c) Mean RMSE vs Sample Size by Zone', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')

plt.suptitle('Detailed Statistics for 36 IMD Climatic Zones', fontsize=14, fontweight='bold')
plt.savefig(os.path.join(OUTPUT_DIR, PLOT_ZONES_STATS), dpi=300, bbox_inches='tight')
print(f"  ✓ Saved to {PLOT_ZONES_STATS}")

plt.show()

print("\n" + "="*80)
print("✓ RMSE ANALYSIS WITH 36 IMD ZONES COMPLETE")
print("="*80)
print(f"\nOutput files:")
print(f"  - {PLOT_MAIN} (Main 4-panel figure)")
print(f"  - {PLOT_ZONES_STATS} (All 36 zones detailed statistics)")
print(f"  - {STATS_FILE} (Comprehensive statistics)")
print(f"  - {ZONES_CSV} (Zone statistics table)")
print()

In [ ]:
#!/usr/bin/env python3
"""
IMD ZONES RMSE RANKING — TEST PERIOD (2009–2014)
Single-panel bar plot with large, legible fonts (Q1-ready)
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import rioxarray
import geopandas as gpd

# ============================================================
# PATHS & CONFIG
# ============================================================

HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM  = "<SET YOUR PATH HERE>"
IMD_ZONES_SHP = "<SET YOUR PATH HERE>"

OUTPUT_PNG = "<SET YOUR PATH HERE>"

# Test period (YYYYDOY)
TEST_START = 2009219
TEST_END   = 2014365

# =========================
# FONT SETTINGS (LARGE)
# =========================
TITLE_FS   = 22
LABEL_FS   = 20
TICK_FS    = 16
LEGEND_FS  = 16
ANNOT_FS   = 16

plt.rcParams.update({
    "axes.titlesize": TITLE_FS,
    "axes.labelsize": LABEL_FS,
    "xtick.labelsize": TICK_FS,
    "ytick.labelsize": TICK_FS,
    "legend.fontsize": LEGEND_FS,
})

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def filter_period(files, start=TEST_START, end=TEST_END):
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except Exception:
            continue
    return valid

# ============================================================
# LOAD & FILTER FILE LISTS
# ============================================================

f_mswx_all = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
f_ddpm_all = sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))

f_mswx = filter_period(f_mswx_all)
f_ddpm = filter_period(f_ddpm_all)

print(f"Filtered to test period:")
print(f"  MSWX files: {len(f_mswx)}")
print(f"  DDPM files: {len(f_ddpm)}")

if len(f_mswx) == 0 or len(f_ddpm) == 0:
    raise RuntimeError("No files found in test period.")

# ============================================================
# OPEN DATASETS
# ============================================================

ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

da_m = ds_mswx["air_temperature"]
da_d = ds_ddpm["air_temperature"]

# ============================================================
# COMPUTE RMSE (PER GRID CELL)
# ============================================================

bias_ts = da_d - da_m
rmse = np.sqrt((bias_ts ** 2).mean("time", skipna=True))

rmse = rmse.rename({"lat": "y", "lon": "x"})
rmse.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
rmse.rio.write_crs("EPSG:4326", inplace=True)

# ============================================================
# LOAD IMD ZONES
# ============================================================

gdf_zones = gpd.read_file(IMD_ZONES_SHP)

# ============================================================
# ZONE-WISE RMSE STATS
# ============================================================

zone_stats = {}

for _, row in gdf_zones.iterrows():
    zone_name = row["ST_NM"]

    try:
        zone_clip = rmse.rio.clip(
            [row.geometry],
            gdf_zones.crs,
            drop=True,
            all_touched=True
        )

        vals = zone_clip.values.flatten()
        vals = vals[np.isfinite(vals)]

        if len(vals) > 0:
            zone_stats[zone_name] = {
                "mean": float(np.nanmean(vals)),
                "std": float(np.nanstd(vals)),
                "n": int(len(vals)),
            }

    except Exception:
        continue

zones_df = pd.DataFrame(zone_stats).T.sort_values("mean")

global_mean_rmse = zones_df["mean"].mean()

# ============================================================
# SINGLE-PANEL BAR CHART (LEGIBLE)
# ============================================================

plt.figure(figsize=(20, 8))

zone_names = zones_df.index.tolist()
zone_means = zones_df["mean"].values
zone_stds  = zones_df["std"].values

colors_zones = plt.cm.RdYlGn_r(np.linspace(0, 1, len(zone_names)))

plt.bar(
    range(len(zone_names)),
    zone_means,
    yerr=zone_stds,
    color=colors_zones,
    edgecolor="black",
    linewidth=1.0,
    capsize=5,
    error_kw={"elinewidth": 1.5},
)

plt.xticks(range(len(zone_names)), zone_names, rotation=90, fontsize=TICK_FS)
plt.ylabel("Mean RMSE (°C)", fontsize=LABEL_FS, fontweight="bold")
plt.grid(True, axis="y", alpha=0.3)

# Global mean reference
plt.axhline(
    global_mean_rmse,
    color="black",
    linestyle="--",
    linewidth=2.5,
    label=f"India Mean RMSE: {global_mean_rmse:.3f} °C"
)

plt.legend(loc="upper left", fontsize=LEGEND_FS)
plt.tight_layout()

plt.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✅ IMD zone RMSE ranking (test period) saved at:\n{OUTPUT_PNG}")

In [ ]:
#!/usr/bin/env python3
"""
Spatial Pearson Correlation Analysis - IMD Climatic Zones
Mirrors the RMSE pipeline but computes per-pixel Pearson r between DDPM and MSWX.

Baseline used: persistence (lag-1) correlation computed on the MSWX series.
Author: Adapted for Correlation by ChatGPT
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import glob
import os
import rioxarray
import geopandas as gpd
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION (same as your RMSE script)
# =====================================================================

HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"
IMD_ZONES_SHP = "<SET YOUR PATH HERE>"
OUTPUT_DIR = "<SET YOUR PATH HERE>"

os.makedirs(OUTPUT_DIR, exist_ok=True)

PLOT_MAIN = "corr_spatial_analysis_main_imd.pdf"
PLOT_ZONES_MAP = "corr_imd_zones_map.pdf"
PLOT_ZONES_STATS = "corr_imd_zones_statistics.pdf"
STATS_FILE = "corr_analysis_statistics_imd.txt"
ZONES_CSV = "corr_imd_zones_statistics.csv"

# =====================================================================
# HELPERS (date handling copied from your script)
# =====================================================================

def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365):
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

# Robust per-pixel Pearson r for (time, y, x) DataArrays
def compute_spatial_pearson(da_model, da_obs):
    """
    Compute Pearson r per pixel between da_model and da_obs along 'time'.
    Returns three xarray DataArrays: (pearson_r, p_value, n_valid) with spatial dims restored.
    """
    da_model, da_obs = xr.align(da_model, da_obs)

    if 'time' not in da_model.dims:
        raise ValueError("Input DataArrays must have 'time' dimension")

    # Stack spatial dims to 'points' -> shape (time, points)
    stacked_model = da_model.stack(points=("y", "x")).transpose("time", "points")
    stacked_obs   = da_obs.stack(points=("y", "x")).transpose("time", "points")

    arr_model = stacked_model.values  # shape (T, N)
    arr_obs   = stacked_obs.values

    T, N = arr_model.shape
    r_vals = np.full(N, np.nan, dtype=np.float32)
    p_vals = np.full(N, np.nan, dtype=np.float32)
    valid_counts = np.zeros(N, dtype=int)

    # compute per-column correlation (loop; safe but can be slow for very large N)
    for j in range(N):
        a = arr_model[:, j]
        b = arr_obs[:, j]
        mask = np.isfinite(a) & np.isfinite(b)
        n = mask.sum()
        valid_counts[j] = int(n)
        if n > 2:
            try:
                r, p = stats.pearsonr(a[mask], b[mask])
            except Exception:
                r, p = np.nan, np.nan
            r_vals[j] = r
            p_vals[j] = p

    # IMPORTANT: obtain a proper MultiIndex for 'points' so unstack works
    points_index = stacked_model['points'].to_index()  # <-- this is a pandas.MultiIndex

    # Create DataArrays and unstack back to (y, x)
    da_r = xr.DataArray(data=r_vals, coords={"points": points_index}, dims=["points"]).unstack("points").rename("pearson_r")
    da_p = xr.DataArray(data=p_vals, coords={"points": points_index}, dims=["points"]).unstack("points").rename("p_value")
    da_n = xr.DataArray(data=valid_counts, coords={"points": points_index}, dims=["points"]).unstack("points").rename("n_valid")

    return da_r, da_p, da_n

# Compute persistence baseline: correlation between obs(t) and obs(t-1)
def compute_persistence_correlation(da_obs):
    da_shift = da_obs.shift(time=1)
    # align to remove NaN introduced by shift
    da_obs_a, da_shift_a = xr.align(da_obs, da_shift)
    r_pers, p_pers, n_pers = compute_spatial_pearson(da_shift_a, da_obs_a)
    return r_pers

# Fisher z transform helpers for averaging correlations
def fisher_z(r):
    r = np.asarray(r)
    # clip to valid range avoid inf
    r = np.clip(r, -0.999999, 0.999999)
    return 0.5 * np.log((1 + r) / (1 - r))

def inverse_fisher_z(z):
    return (np.exp(2*z) - 1) / (np.exp(2*z) + 1)

# =====================================================================
# MAIN
# =====================================================================

print("="*80)
print("CORRELATION SPATIAL ANALYSIS - IMD CLIMATIC ZONES (36 ZONES)")
print("="*80)
print()

# 1. LOAD DATA
print("Step 1: Loading data...")
f_mswx = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))

f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"  Found {len(f_mswx)} MSWX files")
print(f"  Found {len(f_ddpm)} DDPM files")

ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# 2. ASSIGN TIME
print("\nStep 2: Assigning time coordinates...")

if "time" not in ds_mswx.coords:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

ds_mswx["time"] = pd.to_datetime(ds_mswx["time"].values)
ds_ddpm["time"] = pd.to_datetime(ds_ddpm["time"].values)

# 3. COMPUTE PER-PIXEL PEARSON R
print("\nStep 3: Computing per-pixel Pearson correlation (DDPM vs MSWX)...")
da_ddpm = ds_ddpm["air_temperature"]
da_mswx = ds_mswx["air_temperature"]

# Ensure dims name consistency (rename lat/lon -> y/x for rioxarray later)
if "lat" in da_ddpm.dims:
    da_ddpm = da_ddpm.rename({"lat":"y", "lon":"x"})
    da_mswx = da_mswx.rename({"lat":"y", "lon":"x"})

pearson_r, pearson_p, pearson_n = compute_spatial_pearson(da_ddpm, da_mswx)

print("  Correlation computed.")

# 4. PERSISTENCE BASELINE
print("Step 4: Computing persistence baseline (lag-1) correlation on MSWX...")
r_persistence = compute_persistence_correlation(da_mswx)
print("  Persistence baseline computed.")

# 5. PREPARE SPATIAL (rioxarray / clip)
print("\nStep 5: Preparing spatial data for plotting and zone analysis...")
pearson_r.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
pearson_r.rio.write_crs("EPSG:4326", inplace=True)
r_persistence.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
r_persistence.rio.write_crs("EPSG:4326", inplace=True)

gdf_india = gpd.read_file(INDIA_SHP)
pearson_r_clipped = pearson_r.rio.clip(gdf_india.geometry, gdf_india.crs)
r_persistence_clipped = r_persistence.rio.clip(gdf_india.geometry, gdf_india.crs)

gdf_zones = gpd.read_file(IMD_ZONES_SHP)
print(f"  Found {len(gdf_zones)} zones")

lat = pearson_r_clipped.y.values
lon = pearson_r_clipped.x.values
print(f"  Domain: {len(lat)} × {len(lon)} grid points")

# 6. GLOBAL STATS
print("\nStep 6: Computing global statistics...")
r_vals = pearson_r_clipped.values
r_valid = r_vals[~np.isnan(r_vals)]
global_stats = {
    'mean': float(np.nanmean(r_vals)),
    'median': float(np.nanmedian(r_vals)),
    'std': float(np.nanstd(r_vals)),
    'min': float(np.nanmin(r_vals)),
    'max': float(np.nanmax(r_vals)),
    'p25': float(np.nanpercentile(r_vals, 25)),
    'p75': float(np.nanpercentile(r_vals, 75)),
    'n_valid': int(np.sum(~np.isnan(r_vals))),
}
# Baseline global mean (use Fisher z to average correlations properly)
pers_vals = r_persistence_clipped.values
# compute mean correlation using Fisher z transform for both model and persistence
z_model = fisher_z(r_vals[~np.isnan(r_vals)])
z_pers = fisher_z(pers_vals[~np.isnan(pers_vals)])
mean_r_model = inverse_fisher_z(np.nanmean(z_model)) if z_model.size>0 else np.nan
mean_r_pers = inverse_fisher_z(np.nanmean(z_pers)) if z_pers.size>0 else np.nan
improvement = (mean_r_model - mean_r_pers)  # difference in mean r (Fisher-averaged)

print(f"\nGlobal Correlation Statistics:")
print(f"  Mean r (Fisher-avg): {mean_r_model:.4f}")
print(f"  Median r:            {global_stats['median']:.4f}")
print(f"  Std r:               {global_stats['std']:.4f}")

print(f"\nBaseline (persistence) Mean r (Fisher-avg): {mean_r_pers:.4f}")
print(f"Improvement (Δr): {improvement:.4f}")

# 7. ZONES ANALYSIS (36 IMD zones)
print("\nStep 7: Computing statistics for 36 IMD zones...")
zone_stats = {}
for idx, zone_row in gdf_zones.iterrows():
    zone_name = zone_row['ST_NM']
    try:
        zone_subset = pearson_r_clipped.rio.clip([zone_row.geometry], gdf_zones.crs, drop=True, all_touched=True)
        vals = zone_subset.values.flatten()
        vals_valid = vals[~np.isnan(vals)]
        if len(vals_valid) > 0:
            # Fisher-average the correlations inside the zone for a robust mean r
            z = fisher_z(vals_valid)
            mean_r_zone = inverse_fisher_z(np.nanmean(z))
            zone_stats[zone_name] = {
                'mean': float(mean_r_zone),
                'std': float(np.nanstd(vals_valid)),
                'min': float(np.nanmin(vals_valid)),
                'max': float(np.nanmax(vals_valid)),
                'median': float(np.nanmedian(vals_valid)),
                'p25': float(np.nanpercentile(vals_valid, 25)),
                'p75': float(np.nanpercentile(vals_valid, 75)),
                'n_valid': int(len(vals_valid)),
                'cat': int(zone_row['cat']) if 'cat' in zone_row else 0,
            }
            print(f"  {zone_name:<30}: mean r = {zone_stats[zone_name]['mean']:.3f} (n={zone_stats[zone_name]['n_valid']:,})")
    except Exception as e:
        print(f"  Skipping {zone_name}: {e}")
        continue

print(f"\n✓ Statistics computed for {len(zone_stats)} zones")

# 8. STATISTICAL TESTING (Kruskal-Wallis on correlations)
print("\nStep 8: Running Kruskal-Wallis across zones...")
zones_data = []
zones_names = []
for zone_name in sorted(zone_stats.keys()):
    try:
        zone_geom = gdf_zones[gdf_zones['ST_NM'] == zone_name].geometry.values
        zone_clip = pearson_r_clipped.rio.clip(zone_geom, gdf_zones.crs, drop=True, all_touched=True)
        vals = zone_clip.values.flatten()
        vals_clean = vals[~np.isnan(vals)]
        if len(vals_clean) > 0:
            zones_data.append(vals_clean)
            zones_names.append(zone_name)
    except Exception:
        continue

if len(zones_data) > 1:
    h_stat, p_val = stats.kruskal(*zones_data)
    print(f"\nKruskal-Wallis Test (IMD Zone Differences):")
    print(f"  H-statistic: {h_stat:.4f}")
    print(f"  P-value:     {p_val:.2e}")
    print(f"  Result:      {'Significant' if p_val < 0.05 else 'Not significant'}")
else:
    print("\nInsufficient data for Kruskal-Wallis.")
    h_stat, p_val = np.nan, np.nan

# 9. SAVE STATISTICS
print("\nStep 9: Saving statistics...")
try:
    with open(os.path.join(OUTPUT_DIR, STATS_FILE), 'w') as f:
        f.write("="*80 + "\n")
        f.write("PEARSON CORRELATION SPATIAL ANALYSIS - 36 IMD CLIMATIC ZONES\n")
        f.write("="*80 + "\n\n")
        f.write(f"Analysis Period: 2009-2014\n")
        f.write(f"Domain: {len(lat)} × {len(lon)} grid points\n")
        f.write(f"Valid grid cells: {global_stats['n_valid']:,}\n")
        f.write(f"Number of zones: {len(zone_stats)}\n\n")
        f.write("GLOBAL CORRELATION STATISTICS\n")
        f.write("-"*80 + "\n")
        f.write(f"Mean r (Fisher-avg): {mean_r_model:.4f}\n")
        for key, val in global_stats.items():
            f.write(f"{key:<15}: {val}\n")
        f.write("\n\nBASELINE (Persistence)\n")
        f.write("-"*80 + "\n")
        f.write(f"Persistence mean r (Fisher-avg): {mean_r_pers:.4f}\n")
        f.write(f"Δr (model - persistence): {improvement:.4f}\n\n")
        f.write("\n\nIMD ZONE STATISTICS (36 ZONES)\n")
        f.write("-"*80 + "\n")
        f.write(f"{'Zone Name':<30} {'Mean r':<10} {'Std':<10} {'Median':<10} {'N Valid':<10}\n")
        f.write("-"*80 + "\n")
        for zone_name in sorted(zone_stats.keys()):
            stats_dict = zone_stats[zone_name]
            f.write(f"{zone_name:<30} "
                   f"{stats_dict['mean']:<10.4f} "
                   f"{stats_dict['std']:<10.4f} "
                   f"{stats_dict['median']:<10.4f} "
                   f"{stats_dict['n_valid']:<10}\n")
        f.write("\n\nSTATISTICAL TESTS\n")
        f.write("-"*80 + "\n")
        f.write(f"Kruskal-Wallis Test (IMD Zone Differences):\n")
        f.write(f"  H-statistic: {h_stat:.4f}\n")
        f.write(f"  P-value:     {p_val:.2e}\n")
        f.write(f"  Number of zones: {len(zone_stats)}\n")
    print(f"  ✓ Statistics saved to {STATS_FILE}")
except Exception as e:
    print(f"  ✗ Error saving: {e}")

# Save zones CSV
try:
    zones_df = pd.DataFrame(zone_stats).T.sort_values('mean', ascending=False)
    zones_df.to_csv(os.path.join(OUTPUT_DIR, ZONES_CSV))
    print(f"  ✓ Zone statistics saved to {ZONES_CSV}")
except Exception as e:
    print(f"  ✗ Error saving zones CSV: {e}")

# =====================================================================
# PLOTTING (mirrors your RMSE plotting layout)
# =====================================================================

print("\nStep 10: Creating publication-quality figures...\n")
# FIGURE 1: MAIN 4-PANEL
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.25)

# Panel a: correlation map
ax1 = fig.add_subplot(gs[0, 0])
im1 = pearson_r_clipped.plot(ax=ax1, vmin=-1, vmax=1, cmap="RdBu_r", add_colorbar=False, x='x', y='y')
gdf_india.boundary.plot(ax=ax1, edgecolor='black', linewidth=1)
ax1.set_xlabel('Longitude (°E)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Latitude (°N)', fontsize=11, fontweight='bold')
ax1.set_title('(a) Spatial Pearson r (DDPM vs MSWX)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
ax1.tick_params(labelsize=10)

stats_text = (f"Mean r (Fisher-avg): {mean_r_model:.3f}\n"
              f"Median r: {global_stats['median']:.3f}\n"
              f"Std: {global_stats['std']:.3f}\n"
              f"N valid: {global_stats['n_valid']:,}")
ax1.text(0.98, 0.98, stats_text, transform=ax1.transAxes,
         fontsize=9, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

# Panel b: histogram
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(r_valid, bins=50, edgecolor='black', alpha=0.7)
ax2.axvline(mean_r_model, color='red', linestyle='--', linewidth=2, label=f"Mean r (F): {mean_r_model:.3f}")
ax2.set_xlabel('Pearson r', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax2.set_title('(b) Correlation Distribution', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')
ax2.tick_params(labelsize=10)

# Panel c: top 15 zones by mean r (descending)
ax3 = fig.add_subplot(gs[1, 0])
top_zones = sorted(zone_stats.items(), key=lambda x: x[1]['mean'], reverse=True)[:15]
zone_names_top = [z[0][:20] for z in top_zones]
zone_means_top = [z[1]['mean'] for z in top_zones]
zone_stds_top = [z[1]['std'] for z in top_zones]

ax3.barh(range(len(zone_names_top)), zone_means_top, xerr=zone_stds_top)
ax3.set_yticks(range(len(zone_names_top)))
ax3.set_yticklabels(zone_names_top, fontsize=8)
ax3.set_xlabel('Mean Pearson r', fontsize=11, fontweight='bold')
ax3.set_title('(c) Top 15 IMD Zones by Mean r', fontsize=12, fontweight='bold')
ax3.invert_yaxis()
ax3.grid(True, alpha=0.3, axis='x')

# Panel d: comparison with persistence baseline
ax4 = fig.add_subplot(gs[1, 1])
methods = ['Persistence', 'DDPM']
means = [mean_r_pers, mean_r_model]
bars = ax4.bar(methods, means, color=['lightgray', 'lightblue'], edgecolor='black')
ax4.set_ylabel('Mean Pearson r (Fisher-averaged)', fontsize=11, fontweight='bold')
ax4.set_title(f'(d) Baseline Comparison (Δr = {improvement:.3f})', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, means):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

cbar_ax = fig.add_axes([0.22, 0.08, 0.15, 0.01])
cbar = fig.colorbar(im1, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Pearson r', fontsize=10, fontweight='bold')

plt.suptitle('Pearson Correlation Spatial Analysis with 36 IMD Zones: DDPM vs MSWX (2009-2014)', fontsize=14, fontweight='bold', y=0.995)
plt.savefig(os.path.join(OUTPUT_DIR, PLOT_MAIN), dpi=300, bbox_inches='tight')
print(f"  ✓ Saved to {PLOT_MAIN}")

# FIGURE 2: All 36 zones stats (mirrors your RMSE figure)
fig2 = plt.figure(figsize=(18, 12))
gs2 = GridSpec(2, 2, figure=fig2, hspace=0.4, wspace=0.25)

ax = fig2.add_subplot(gs2[0, :])
all_zones = sorted(zone_stats.items(), key=lambda x: x[1]['mean'], reverse=True)
zone_names_all = [z[0] for z in all_zones]
zone_means_all = [z[1]['mean'] for z in all_zones]
zone_stds_all = [z[1]['std'] for z in all_zones]
colors_zones = plt.cm.RdYlBu(np.linspace(0, 1, len(all_zones)))
bars = ax.bar(range(len(zone_names_all)), zone_means_all, yerr=zone_stds_all,
             color=colors_zones, edgecolor='black', linewidth=0.5, capsize=3)
ax.set_xticks(range(len(zone_names_all)))
ax.set_xticklabels(zone_names_all, rotation=90, fontsize=7)
ax.set_ylabel('Mean Pearson r', fontsize=11, fontweight='bold')
ax.set_title('(a) All 36 IMD Zones Ranked by Mean r with Std Dev', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(mean_r_model, color='red', linestyle='--', linewidth=2, label=f"Global mean r (F): {mean_r_model:.3f}")
ax.legend(fontsize=10)

ax = fig2.add_subplot(gs2[1, 0])
zone_stds = [zone_stats[z]['std'] for z in zone_names_all]
ax.scatter(zone_means_all, zone_stds, s=100, alpha=0.6, edgecolors='black')
ax.set_xlabel('Mean Pearson r', fontsize=11, fontweight='bold')
ax.set_ylabel('Standard Deviation', fontsize=11, fontweight='bold')
ax.set_title('(b) Mean vs Std Dev by Zone', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = fig2.add_subplot(gs2[1, 1])
zone_n_valid = [zone_stats[z]['n_valid'] for z in zone_names_all]
ax.scatter(zone_means_all, zone_n_valid, s=100, alpha=0.6, edgecolors='black')
ax.set_xlabel('Mean Pearson r', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Valid Grid Cells', fontsize=11, fontweight='bold')
ax.set_title('(c) Mean r vs Sample Size by Zone', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')

plt.suptitle('Detailed Correlation Statistics for 36 IMD Climatic Zones', fontsize=14, fontweight='bold')
plt.savefig(os.path.join(OUTPUT_DIR, PLOT_ZONES_STATS), dpi=300, bbox_inches='tight')
print(f"  ✓ Saved to {PLOT_ZONES_STATS}")

plt.show()

print("\n" + "="*80)
print("✓ CORRELATION ANALYSIS WITH 36 IMD ZONES COMPLETE")
print("="*80)
print(f"\nOutput files:")
print(f"  - {PLOT_MAIN} (Main 4-panel figure)")
print(f"  - {PLOT_ZONES_STATS} (All 36 zones detailed statistics)")
print(f"  - {STATS_FILE} (Comprehensive statistics)")
print(f"  - {ZONES_CSV} (Zone statistics table)")
print()


In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import rioxarray
import geopandas as gpd

# -----------------------------
# Paths
# -----------------------------
hr_mswx = "<SET YOUR PATH HERE>"
hr_ddpm = "<SET YOUR PATH HERE>"
india_shp = "<SET YOUR PATH HERE>"

# -----------------------------
# Helper functions
# -----------------------------
def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2003079, end=2014365):
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

# -----------------------------
# Step 1: Load files
# -----------------------------
f_mswx = sorted(glob.glob(os.path.join(hr_mswx, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(hr_ddpm, "*_ddpm.nc")))

f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"Found {len(f_mswx)} MSWX files and {len(f_ddpm)} DDPM files")

# -----------------------------
# Step 2: Open datasets
# -----------------------------
ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# -----------------------------
# Step 3: Assign time if missing
# -----------------------------
if "time" not in ds_mswx:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

ds_mswx["time"] = pd.to_datetime(ds_mswx["time"].values)
ds_ddpm["time"] = pd.to_datetime(ds_ddpm["time"].values)

# -----------------------------
# Step 4: Compute the spatial correlation
# -----------------------------
# Calculate anomalies by subtracting the mean over time
anom_mswx = ds_mswx['air_temperature'] - ds_mswx['air_temperature'].mean(dim='time')
anom_ddpm = ds_ddpm['air_temperature'] - ds_ddpm['air_temperature'].mean(dim='time')

# Calculate the correlation coefficient at each grid point
# This computes the Pearson correlation over the time dimension for each (lat, lon) point
spatial_corr = xr.corr(anom_mswx, anom_ddpm, dim='time')

# -----------------------------
# Step 5: Prepare for rioxarray clipping
# -----------------------------
spatial_corr = spatial_corr.rename({"lat": "y", "lon": "x"})
spatial_corr.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
spatial_corr.rio.write_crs("EPSG:4326", inplace=True)

# Load India shapefile
gdf = gpd.read_file(india_shp)

# Clip to India
spatial_corr_clipped = spatial_corr.rio.clip(gdf.geometry, gdf.crs)

# -----------------------------
# Step 6: Plotting
# -----------------------------
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 10))

# Plot the spatial correlation distribution
spatial_corr_clipped.plot(ax=ax, cmap="viridis", vmin=0.95, vmax=1, cbar_kwargs={'label': "Correlation Coefficient"})

# Overlay India boundary
gdf.boundary.plot(ax=ax, edgecolor="black", linewidth=0.5)

ax.set_title("Temporal Pattern Correlation (DDPM vs. MSWX)", fontsize=16)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import rioxarray
import geopandas as gpd
from scipy import stats 
from statsmodels.tsa.seasonal import seasonal_decompose # For decomposition

# -----------------------------
# Paths
# -----------------------------
hr_mswx = "<SET YOUR PATH HERE>"
hr_ddpm = "<SET YOUR PATH HERE>"
india_shp = "<SET YOUR PATH HERE>"
output_plot_dir = "<SET YOUR PATH HERE>"
os.makedirs(output_plot_dir, exist_ok=True)

# -----------------------------
# Helper functions
# -----------------------------
def get_file_date(fname):
    """Extracts YYYYDOY date from filename."""
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    """Converts YYYYDOY integer to a pandas datetime object."""
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2003079, end=2014365):
    """Filters a list of filenames based on YYYYDOY range."""
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

# -----------------------------
# Step 1: Load files
# -----------------------------
f_mswx = sorted(glob.glob(os.path.join(hr_mswx, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(hr_ddpm, "*_ddpm.nc")))

f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"Found {len(f_mswx)} MSWX files and {len(f_ddpm)} DDPM files")

# -----------------------------
# Step 2: Open datasets
# -----------------------------
ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", 
                            engine="netcdf4", chunks={'time': 'auto'})
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", 
                            engine="netcdf4", chunks={'time': 'auto'})

# -----------------------------
# Step 3: Assign time if missing
# -----------------------------
if "time" not in ds_mswx.coords:
    print("Time coordinate not found, building from filenames...")
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))
else:
    print("Time coordinate found.")

ds_mswx["time"] = pd.to_datetime(ds_mswx["time"].values)
ds_ddpm["time"] = pd.to_datetime(ds_ddpm["time"].values)

# -----------------------------
# Step 4: Compute the spatial anomalies
# -----------------------------
print("Calculating temporal mean...")
mean_mswx = ds_mswx['air_temperature'].mean(dim='time')
mean_ddpm = ds_ddpm['air_temperature'].mean(dim='time')

print("Calculating anomalies...")
anom_mswx = ds_mswx['air_temperature'] - mean_mswx
anom_ddpm = ds_ddpm['air_temperature'] - mean_ddpm

# -----------------------------
# Step 5: Load Shapefile & Clip Anomalies
# -----------------------------
print(f"Loading shapefile from {india_shp}...")
gdf = gpd.read_file(india_shp)

anom_mswx = anom_mswx.rename({"lat": "y", "lon": "x"})
anom_mswx.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
anom_mswx.rio.write_crs("EPSG:4326", inplace=True)
anom_mswx_clipped = anom_mswx.rio.clip(gdf.geometry, gdf.crs)

anom_ddpm = anom_ddpm.rename({"lat": "y", "lon": "x"})
anom_ddpm.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
anom_ddpm.rio.write_crs("EPSG:4326", inplace=True)
anom_ddpm_clipped = anom_ddpm.rio.clip(gdf.geometry, gdf.crs)

print("Anomalies clipped to India boundary.")

# -----------------------------
# Step 6: Calculate the Pattern Correlation (ACC)
# -----------------------------
print("Calculating daily pattern correlation (ACC)...")
pattern_corr_ts = xr.corr(anom_mswx_clipped, anom_ddpm_clipped, dim=['y', 'x'])
pattern_corr_ts = pattern_corr_ts.compute()
print("ACC calculation complete.")

# ----------------------------------------------------
# Step 6.5: Calculate & Print Daily Summary Statistics
# ----------------------------------------------------
print("\n" + "="*70)
print("     SUMMARY: Daily Pattern Correlation (ACC) Statistics")
print(f"     (Period: {str(pattern_corr_ts.time.min().values)[:10]} to {str(pattern_corr_ts.time.max().values)[:10]})")
print("="*70 + "\n")

# Get raw values
acc_values = pattern_corr_ts.values
time_axis = np.arange(len(acc_values))

# Basic stats
mean_acc = float(np.mean(acc_values))
median_acc = float(np.median(acc_values))
std_acc = float(np.std(acc_values))
min_acc = float(np.min(acc_values))
max_acc = float(np.max(acc_values))
pct_gt_99 = float((np.sum(acc_values > 0.99) / len(acc_values)) * 100)

print(f"Mean Daily ACC:   {mean_acc:.6f}")
print(f"Median Daily ACC: {median_acc:.6f}")
print(f"Std Dev of ACC:   {std_acc:.6f}")
print(f"Min Daily ACC:    {min_acc:.6f}")
print(f"Max Daily ACC:    {max_acc:.6f}")
print(f"% Days ACC > 0.99: {pct_gt_99:.2f}%")

# Trend
trend_acc = stats.linregress(time_axis, acc_values)
slope_per_decade = trend_acc.slope * 365.25 * 10
print(f"Trend in Daily ACC: {slope_per_decade:+.6f} per decade (p-value: {trend_acc.pvalue:.3f})")
print("\n" + "="*70 + "\n")


# -----------------------------
# Step 7: Time Series Decomposition
# -----------------------------
print("Resampling daily ACC to monthly and decomposing...")

# Resample to monthly mean ACC
monthly_acc_ts = pattern_corr_ts.resample(time="M").mean()

# Convert to pandas Series for decomposition
acc_series = monthly_acc_ts.to_series().dropna()

# Perform decomposition (period=12 for monthly data)
decomp_acc = seasonal_decompose(acc_series, model='additive', period=12)

print("✅ Decomposition complete")

# ----------------------------------------------------
# Step 7.5: Calculate & Print Decomposition Statistics
# ----------------------------------------------------
print("\n" + "="*70)
print("     SUMMARY: Monthly ACC Decomposition Statistics")
print("="*70 + "\n")

# --- Trend Component Stats ---
trend_component = decomp_acc.trend.dropna()
time_axis_trend = np.arange(len(trend_component))
trend_slope = stats.linregress(time_axis_trend, trend_component).slope
trend_slope_decade = trend_slope * 12 * 10 # °C/month -> °C/decade
print("--- (b) Trend Component ---")
print(f"Mean Trend (ACC):   {np.mean(trend_component):.6f}")
print(f"Trend Slope:        {trend_slope_decade:+.6f} ACC points/decade")
print("\n")

# --- Seasonal Component Stats ---
seasonal_component = decomp_acc.seasonal
# Get one cycle
seasonal_cycle = seasonal_component.values[:12]
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

best_month_idx = np.argmax(seasonal_cycle)
worst_month_idx = np.argmin(seasonal_cycle)
amplitude = seasonal_cycle[best_month_idx] - seasonal_cycle[worst_month_idx]

print("--- (c) Seasonal Component ---")
print(f"Best Month:       {month_names[best_month_idx]} (ACC: {seasonal_cycle[best_month_idx]:+.6f} from trend)")
print(f"Worst Month:      {month_names[worst_month_idx]} (ACC: {seasonal_cycle[worst_month_idx]:+.6f} from trend)")
print(f"Seasonal Amplitude: {amplitude:.6f} (Max - Min)")
print("\n")

# --- Residual Component Stats ---
residual_component = decomp_acc.resid.dropna()
std_dev_residual = np.std(residual_component)
print("--- (d) Residual (Error) Component ---")
print(f"Mean Residual:    {np.mean(residual_component):.6f}")
print(f"Std Dev Residual: {std_dev_residual:.6f} (Monthly ACC noise level)")
print("\n" + "="*70 + "\n")


# -----------------------------
# Step 8: Plot the Decomposition
# -----------------------------
print("Generating decomposition plots...")
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)
fig.suptitle("Time Series Decomposition of Monthly Pattern Correlation (ACC)", fontsize=18, y=1.02)

# --- Plot 1: Original ---
axes[0].plot(decomp_acc.observed, label="Observed Monthly ACC", color="blue", linewidth=2)
axes[0].set_title("Original Monthly Time Series", fontsize=16)
axes[0].set_ylabel("ACC", fontsize=12)
axes[0].legend(fontsize=12)
axes[0].tick_params(axis='y', labelsize=12)
axes[0].text(0.01, 0.95, "(a)", transform=axes[0].transAxes, fontsize=16, fontweight='bold', va='top', ha='left')

# --- Plot 2: Trend ---
axes[1].plot(decomp_acc.trend, label="Trend Component", color="darkgreen", linewidth=2)
axes[1].set_title("Trend Component", fontsize=16)
axes[1].set_ylabel("ACC", fontsize=12)
axes[1].legend(fontsize=12)
axes[1].tick_params(axis='y', labelsize=12)
axes[1].text(0.01, 0.95, "(b)", transform=axes[1].transAxes, fontsize=16, fontweight='bold', va='top', ha='left')

# --- Plot 3: Seasonal ---
axes[2].plot(decomp_acc.seasonal, label="Seasonal Component", color="purple", linewidth=2)
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title("Seasonal Component", fontsize=16)
axes[2].set_ylabel("ACC", fontsize=12)
axes[2].legend(fontsize=12)
axes[2].tick_params(axis='y', labelsize=12)
axes[2].text(0.01, 0.95, "(c)", transform=axes[2].transAxes, fontsize=16, fontweight='bold', va='top', ha='left')

# --- Plot 4: Residual ---
axes[3].plot(decomp_acc.resid, label="Residual", color="red", alpha=0.7, marker='.', markersize=4, linestyle='None')
axes[3].axhline(0, color='black', linestyle='--', linewidth=1)
axes[3].set_title("Residual Component", fontsize=16)
axes[3].set_ylabel("ACC", fontsize=12)
axes[3].set_xlabel("Time", fontsize=14)
axes[3].legend(fontsize=12)
axes[3].tick_params(axis='both', which='major', labelsize=12)
axes[3].text(0.01, 0.95, "(d)", transform=axes[3].transAxes, fontsize=16, fontweight='bold', va='top', ha='left')

plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust for suptitle

# -----------------------------
# Step 9: Save and Show
# -----------------------------
plt.show()

# # Save the figure
# save_path = os.path.join(output_plot_dir, "DDPM_Monthly_ACC_Decomposition.png")
# fig.savefig(save_path, dpi=300, bbox_inches='tight')

# print(f"✅ Plotting complete. Figure saved to {save_path}")

In [ ]:
import xarray as xr
# Raw CMIP6 Model
ds1 = xr.open_dataset("<SET YOUR PATH HERE>")
print(ds1)


In [ ]:
import xarray as xr
ds2 = xr.open_dataset("<SET YOUR PATH HERE>")
print(ds2)

In [ ]:
import xarray as xr
import numpy as np
import glob
import os
from tqdm import tqdm
from datetime import datetime

# ==============================
# CONFIG
# ==============================
RAW_CMIP6_DIR  = "<SET YOUR PATH HERE>"
DDPM_CMIP6_DIR = "<SET YOUR PATH HERE>"
MSWX_DIR       = "<SET YOUR PATH HERE>"

# ==============================
# 1. BUILD MSWX DATE → FILE
#    MSWX filenames: 1979001.nc  (YYYY + DOY)
# ==============================
mswx_files = glob.glob(os.path.join(MSWX_DIR, "*.nc"))
print(f"Total MSWX files: {len(mswx_files)}")

mswx_dict = {}
for f in mswx_files:
    fname = os.path.basename(f)      # e.g. "1979001.nc"
    yday = fname.replace(".nc", "")  # "1979001"
    try:
        date = datetime.strptime(yday, "%Y%j")
    except Exception as e:
        print(f"[MSWX] Skipping (cannot parse date): {fname} | {e}")
        continue
    mswx_dict[date] = f

print(f"MSWX days available: {len(mswx_dict)}")


# ==============================
# 2. HELPER: BUILD MODEL → {date: file}
#    CMIP6 filenames: YYYY-MM-DD_MODEL.nc
# ==============================
def build_model_date_dict(cmip6_dir):
    cmip6_files = glob.glob(os.path.join(cmip6_dir, "*.nc"))
    print(f"\n[INFO] Scanning directory: {cmip6_dir}")
    print(f"Total CMIP6 files: {len(cmip6_files)}")

    model_to_date_files = {}

    for f in cmip6_files:
        fname = os.path.basename(f)  # e.g. "1979-01-01_CESM2-WACCM.nc"

        if "_" not in fname:
            print(f"[CMIP6] Skipping (no '_'): {fname}")
            continue

        date_part, model_part = fname.split("_", 1)
        model_name = model_part.replace(".nc", "")

        try:
            date = datetime.strptime(date_part, "%Y-%m-%d")
        except Exception as e:
            print(f"[CMIP6] Skipping (bad date '{date_part}'): {fname} | {e}")
            continue

        if model_name not in model_to_date_files:
            model_to_date_files[model_name] = {}

        model_to_date_files[model_name][date] = f

    detected_models = list(model_to_date_files.keys())
    print(f"Detected models: {detected_models}")
    for m in detected_models:
        print(f"  {m}: {len(model_to_date_files[m])} days with files")

    return model_to_date_files


# ==============================
# 3. METRICS FOR ONE MODEL
# ==============================
def compute_metrics_for_model(model_name, date_file_dict, mswx_dict):
    # intersect dates with MSWX (handles calendar differences automatically)
    common_dates = sorted(set(date_file_dict.keys()) & set(mswx_dict.keys()))
    print(f"\nModel: {model_name} | Matched days with MSWX: {len(common_dates)}")

    if len(common_dates) == 0:
        print(f"[{model_name}] No overlapping dates with MSWX. Skipping model.")
        return None

    sum_mae = 0.0
    sum_rmse = 0.0
    sum_bias = 0.0
    sum_corr = 0.0
    sum_std_err = 0.0
    sum_std_ratio = 0.0
    sum_nse = 0.0
    n_days = 0

    for date in tqdm(common_dates, desc=model_name, leave=False):
        f_cmip = date_file_dict[date]
        f_mswx = mswx_dict[date]

        try:
            ds1 = xr.open_dataset(f_cmip)
            ds2 = xr.open_dataset(f_mswx)
        except Exception as e:
            print(f"[{model_name} {date.date()}] Failed to open files, skipping: {e}")
            continue

        # ---- READ CMIP6 tas (handle both (lat, lon) and (time, lat, lon)) ----
        try:
            da_cmip = ds1["tas"]
        except Exception as e:
            print(f"[{model_name} {date.date()}] 'tas' not found, skipping day: {e}")
            ds1.close()
            ds2.close()
            continue

        try:
            if "time" in da_cmip.dims:
                if da_cmip.sizes["time"] != 1:
                    print(
                        f"[{model_name} {date.date()}] Unexpected time size "
                        f"{da_cmip.sizes['time']} in tas, taking time=0 anyway."
                    )
                da_cmip2 = da_cmip.isel(time=0)
            else:
                da_cmip2 = da_cmip

            cmip = da_cmip2.values.astype(np.float64)   # (lat, lon)
        except Exception as e:
            print(f"[{model_name} {date.date()}] Error handling tas dims, skipping: {e}")
            ds1.close()
            ds2.close()
            continue

        # ---- READ MSWX air_temperature(time, lat, lon) ----
        try:
            obs = ds2["air_temperature"].isel(time=0).values.astype(np.float64)
        except Exception as e:
            print(f"[{model_name} {date.date()}] MSWX variable error, skipping: {e}")
            ds1.close()
            ds2.close()
            continue

        # Shape mismatch → skip this date
        if cmip.shape != obs.shape:
            print(
                f"[{model_name} {date.date()}] Shape mismatch, "
                f"CMIP6 {cmip.shape} vs MSWX {obs.shape}. Skipping."
            )
            ds1.close()
            ds2.close()
            continue

        mask = np.isfinite(cmip) & np.isfinite(obs)
        cmip_v = cmip[mask]
        obs_v  = obs[mask]

        if cmip_v.size == 0:
            print(f"[{model_name} {date.date()}] No valid grid points after masking. Skipping.")
            ds1.close()
            ds2.close()
            continue

        err = cmip_v - obs_v

        mae  = np.mean(np.abs(err))
        rmse = np.sqrt(np.mean(err**2))
        bias = np.mean(err)
        std_err = np.std(err)

        # correlation
        try:
            corr = np.corrcoef(cmip_v, obs_v)[0, 1]
        except Exception:
            corr = np.nan

        # std ratio
        std_obs  = np.std(obs_v)
        std_cmip = np.std(cmip_v)
        std_ratio = np.nan
        if std_obs > 0:
            std_ratio = std_cmip / std_obs

        # NSE
        denom = np.sum((obs_v - np.mean(obs_v))**2)
        if denom == 0:
            nse = np.nan
        else:
            nse = 1 - (np.sum(err**2) / denom)

        # accumulate only finite values
        if np.isfinite(mae):       sum_mae += mae
        if np.isfinite(rmse):      sum_rmse += rmse
        if np.isfinite(bias):      sum_bias += bias
        if np.isfinite(corr):      sum_corr += corr
        if np.isfinite(std_err):   sum_std_err += std_err
        if np.isfinite(std_ratio): sum_std_ratio += std_ratio
        if np.isfinite(nse):       sum_nse += nse

        n_days += 1

        ds1.close()
        ds2.close()

    if n_days == 0:
        print(f"[{model_name}] No valid days with metrics computed. Skipping.")
        return None

    mean_metrics = {
        "Model": model_name,
        "Matched Days": n_days,
        "Mean Correlation (r)":        sum_corr / n_days,
        "Mean MAE (K)":                sum_mae / n_days,
        "Mean RMSE (K)":               sum_rmse / n_days,
        "Mean Bias (K)":               sum_bias / n_days,
        "Mean Error Std Dev (K)":      sum_std_err / n_days,
        "Mean Std Ratio (Model/Obs)":  sum_std_ratio / n_days,
        "Mean NSE":                    sum_nse / n_days,
    }
    return mean_metrics


# ==============================
# 4. RUN EVALUATION FOR A DIRECTORY
# ==============================
def evaluate_directory(cmip6_dir, mswx_dict):
    model_to_date_files = build_model_date_dict(cmip6_dir)
    metrics_by_model = {}

    for model_name, date_file_dict in model_to_date_files.items():
        if len(date_file_dict) == 0:
            print(f"\n[{model_name}] No files found for this model. Skipping.")
            continue

        metrics = compute_metrics_for_model(model_name, date_file_dict, mswx_dict)
        if metrics is not None:
            metrics_by_model[model_name] = metrics

    return metrics_by_model


# ==============================
# 5. RUN FOR RAW AND DDPM, THEN COMBINE
# ==============================
raw_metrics  = evaluate_directory(RAW_CMIP6_DIR, mswx_dict)
ddpm_metrics = evaluate_directory(DDPM_CMIP6_DIR, mswx_dict)

# Models that exist in both
common_models = sorted(set(raw_metrics.keys()) & set(ddpm_metrics.keys()))
print("\nCommon models (RAW & DDPM):", common_models)

combined = []

for model in common_models:
    raw  = raw_metrics[model]
    ddpm = ddpm_metrics[model]

    combined.append({
        "Model": model,
        "RMSE_RAW":  raw["Mean RMSE (K)"],
        "RMSE_DDPM": ddpm["Mean RMSE (K)"],
        "MAE_RAW":   raw["Mean MAE (K)"],
        "MAE_DDPM":  ddpm["Mean MAE (K)"],
        "Bias_RAW":  raw["Mean Bias (K)"],
        "Bias_DDPM": ddpm["Mean Bias (K)"],
        "r_RAW":     raw["Mean Correlation (r)"],
        "r_DDPM":    ddpm["Mean Correlation (r)"],
        "NSE_RAW":   raw["Mean NSE"],
        "NSE_DDPM":  ddpm["Mean NSE"],
    })

print("\n========== RAW vs DDPM COMPARISON (ARRAY OF DICTS) ==========\n")
for row in combined:
    print(row)


In [ ]:
import xarray as xr

# Define the base path and model names
base_path = "<SET YOUR PATH HERE>"

models = {
    "CESM2-WACCM": "1979-01-01_CESM2-WACCM.nc",
    "CESM2": "1979-01-01_CESM2.nc",
    "EC-Earth3-AerChem": "1979-01-01_EC-Earth3-AerChem.nc",
    "EC-Earth3-CC": "1979-01-01_EC-Earth3-CC.nc",
    "HadGEM3-GC31-LL": "1979-01-01_HadGEM3-GC31-LL.nc",
    "HadGEM3-GC31-MM": "1979-01-01_HadGEM3-GC31-MM.nc",
    "INM-CM4-8": "1979-01-01_INM-CM4-8.nc",
    "INM-CM5-0": "1979-01-01_INM-CM5-0.nc",
    "MPI-ESM1-2-HR": "1979-01-01_MPI-ESM1-2-HR.nc",
    "NorESM2-MM": "1979-01-01_NorESM2-MM.nc",
}

# Inspect each model
for model_name, filename in models.items():
    filepath = base_path + filename
    ds = xr.open_dataset(filepath)
    
    print("\n" + "="*80)
    print(f"MODEL: {model_name}")
    print("="*80)
    
    # Print dimensions
    print("\nDimensions:")
    print(ds.dims)
    
    # Print full info
    print("\nFull Dataset Info:")
    ds.info()
    
    # Print data variables details
    print("\nData Variables Details:")
    for var in ds.data_vars:
        print(f"  {var}: dims={ds[var].dims}, shape={ds[var].shape}, dtype={ds[var].dtype}")
    
    # Print coordinates details
    print("\nCoordinates Details:")
    for coord in ds.coords:
        print(f"  {coord}: dims={ds[coord].dims}, shape={ds[coord].shape}, dtype={ds[coord].dtype}")
    
    # Close dataset to free memory
    ds.close()

# =====================================================================================
# EXTRA INSPECTIONS (ADDED BY REQUEST)
# =====================================================================================

extra_files = {
    "Ensemble (Top10 Raw)": "<SET YOUR PATH HERE>",
    "MSWX High-Res Daily": "<SET YOUR PATH HERE>"
}

for label, filepath in extra_files.items():
    print("\n" + "="*80)
    print(f"EXTRA FILE: {label}")
    print("="*80)
    try:
        ds = xr.open_dataset(filepath)
    except Exception as e:
        print(f"  ERROR opening {filepath}: {e}")
        continue
    
    # Print dimensions
    print("\nDimensions:")
    print(ds.dims)

    # Full dataset info
    print("\nFull Dataset Info:")
    ds.info()

    # Data variables
    print("\nData Variables Details:")
    for var in ds.data_vars:
        print(f"  {var}: dims={ds[var].dims}, shape={ds[var].shape}, dtype={ds[var].dtype}")

    # Coordinates
    print("\nCoordinates Details:")
    for coord in ds.coords:
        print(f"  {coord}: dims={ds[coord].dims}, shape={ds[coord].shape}, dtype={ds[coord].dtype}")

    ds.close()

print("\n" + "="*80)
print("INSPECTION COMPLETE")
print("="*80)

In [ ]:
#!/usr/bin/env python3
"""
Compute mean temperature inside India for multiple datasets (models, ensemble, MSWX).
Robust handling of time coordinate/dimension, variable naming, and ensemble/model dims.

Output:
 - Printed mean temperatures
 - Saved bar plot at output_dir/mean_temp_by_dataset_india.png
"""

import os
from pathlib import Path
import xarray as xr
import rioxarray
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------ User paths ------------------
model_base = "<SET YOUR PATH HERE>"
models = {
    "CESM2-WACCM": "1979-01-01_CESM2-WACCM.nc",
    "CESM2": "1979-01-01_CESM2.nc",
    "EC-Earth3-AerChem": "1979-01-01_EC-Earth3-AerChem.nc",
    "EC-Earth3-CC": "1979-01-01_EC-Earth3-CC.nc",
    "HadGEM3-GC31-LL": "1979-01-01_HadGEM3-GC31-LL.nc",
    "HadGEM3-GC31-MM": "1979-01-01_HadGEM3-GC31-MM.nc",
    "INM-CM4-8": "1979-01-01_INM-CM4-8.nc",
    "INM-CM5-0": "1979-01-01_INM-CM5-0.nc",
    "MPI-ESM1-2-HR": "1979-01-01_MPI-ESM1-2-HR.nc",
    "NorESM2-MM": "1979-01-01_NorESM2-MM.nc",
}

extra_files = {
    "Ensemble": "<SET YOUR PATH HERE>",
    "MSWX_1979001": "<SET YOUR PATH HERE>",
}

india_shp = "<SET YOUR PATH HERE>"

output_dir = "<SET YOUR PATH HERE>"
os.makedirs(output_dir, exist_ok=True)

# ------------------ Helper functions ------------------
def find_temp_var(ds, candidates=("tas","air_temperature","t2m")):
    for v in candidates:
        if v in ds.data_vars:
            return v
    # fallback: first float var with >=2 dims
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating) and len(ds[v].dims) >= 2:
            return v
    raise RuntimeError("No suitable temperature-like variable found in dataset.")

def ensure_time_and_dims(ds, preferred_time=None, target_var=None, keep_2d_target_if_no_time=True):
    """
    Robustly ensure dataset has:
      - a 'time' dimension (convert existing time coord -> dim if needed) OR keep target 2D
      - lat/lon coords named 'lat' and 'lon'
    If target_var has no time dim and keep_2d_target_if_no_time is True, we DO NOT expand it.
    """
    ds = ds.copy()

    # rename common coordinate names to lat/lon
    coord_map = {}
    for c in list(ds.coords):
        cl = c.lower()
        if cl in ("latitude",) and c != "lat":
            coord_map[c] = "lat"
        if cl in ("longitude",) and c != "lon":
            coord_map[c] = "lon"
        if cl == "y" and c != "lat":
            coord_map[c] = "lat"
        if cl == "x" and c != "lon":
            coord_map[c] = "lon"
    if coord_map:
        ds = ds.rename(coord_map)

    # CASE A: 'time' is a dimension -> coerce dtype if necessary
    if "time" in ds.dims:
        try:
            if not np.issubdtype(ds["time"].dtype, np.datetime64):
                ds["time"] = pd.to_datetime(ds["time"].values)
        except Exception:
            pass

    # CASE B: 'time' exists as coord but not dimension -> convert
    elif "time" in ds.coords and "time" not in ds.dims:
        # Ensure time_vals is iterable (list-like) for expand_dims
        time_vals = ds["time"].values
        # force to at least 1-d and then to python list
        time_vals = np.atleast_1d(time_vals).tolist()
        # drop the non-dimension 'time' variable and expand dims with the list
        ds = ds.drop_vars("time")
        ds = ds.expand_dims({"time": time_vals})
        try:
            ds["time"] = pd.to_datetime(ds["time"].values)
        except Exception:
            pass

    # CASE C: no 'time' anywhere -> create one
    else:
        tval = pd.to_datetime(preferred_time) if preferred_time is not None else pd.to_datetime("1979-01-01")
        ds = ds.expand_dims({"time": [tval]})

    # Ensure target_var handling:
    if target_var is not None and target_var in ds:
        if "time" not in ds[target_var].dims:
            if keep_2d_target_if_no_time:
                # keep as-is (2D). Do not expand.
                pass
            else:
                # expand variable along time dimension using ds["time"] values
                da = ds[target_var]
                da_expanded = da.expand_dims({"time": ds["time"].values})
                ds[target_var] = da_expanded

    return ds

def set_rio_spatial(da):
    # Ensure DataArray has spatial dims assigned for rioxarray
    da = da.copy()
    # determine spatial dims names
    dims = da.dims
    # prefer lat/lon naming
    if ("lat" in dims) and ("lon" in dims):
        xdim, ydim = "lon", "lat"
    elif ("y" in dims) and ("x" in dims):
        xdim, ydim = "x", "y"
    else:
        # try to infer by dimension size heuristics (not guaranteed)
        if len(dims) >= 2:
            ydim, xdim = dims[-2], dims[-1]
        else:
            raise RuntimeError("Unable to determine spatial dims for rioxarray.")
    try:
        da = da.rio.set_spatial_dims(x_dim=xdim, y_dim=ydim, inplace=False)
    except Exception:
        da = da.rio.set_spatial_dims(x_dim=xdim, y_dim=ydim)
    # set CRS if missing (assume EPSG:4326)
    try:
        if da.rio.crs is None:
            da.rio.write_crs("EPSG:4326", inplace=True)
    except Exception:
        da.rio.write_crs("EPSG:4326", inplace=True)
    return da

def mean_within_polygon(ds, varname, gdf, time_idx=0):
    """
    Clip dataset[varname] to geometry gdf and compute mean over spatial dims.
    Handles either 2D (lat,lon) or 3D (time,lat,lon) vars; if time present uses time_idx.
    Returns scalar mean (float).
    """
    if varname not in ds:
        raise RuntimeError(f"{varname} not found in dataset")

    da = ds[varname]
    # if model dim exists (should be handled before), but if present, average it
    if "model" in da.dims:
        da = da.mean(dim="model", skipna=True)

    # pick time slice if present
    if "time" in da.dims:
        # ensure index is within bounds
        tsize = da.sizes["time"]
        if time_idx >= tsize:
            raise IndexError("time_idx out of range")
        da_time = da.isel(time=time_idx)
    else:
        da_time = da

    # set rioxarray spatial dims & CRS
    da_time = set_rio_spatial(da_time)

    # align CRS
    try:
        gdf2 = gdf.to_crs(da_time.rio.crs)
    except Exception:
        gdf2 = gdf

    # clip (drop=True to shrink bounds; all_touched True to include edge cells)
    clipped = da_time.rio.clip(gdf2.geometry, gdf2.crs, drop=True, all_touched=True)

    # compute mean over spatial dims (detect lat/lon or x/y)
    spatial_dims = [d for d in clipped.dims if d in ("lat","lon","y","x")]
    if not spatial_dims:
        raise RuntimeError("No spatial dims found after clipping.")
    mean_val = float(clipped.mean(dim=spatial_dims, skipna=True).values)
    return mean_val

# ------------------ Load India shapefile ------------------
if not os.path.exists(india_shp):
    raise FileNotFoundError(f"India shapefile not found at: {india_shp}")

gdf_india = gpd.read_file(india_shp)
if gdf_india.empty:
    raise RuntimeError("India shapefile loaded but contains no geometry.")

# ------------------ Loop through files and compute means ------------------
results = {}  # store name -> mean_temp

# process model files
for name, fname in models.items():
    fp = os.path.join(model_base, fname)
    if not os.path.exists(fp):
        print(f"Warning: {fp} does not exist — skipping {name}")
        continue

    # open dataset (decode_times=False is safer for odd time encodings)
    ds = xr.open_dataset(fp, decode_times=False)
    # try to convert time coord to datetimes if present
    if "time" in ds.coords and not np.issubdtype(ds["time"].dtype, np.datetime64):
        try:
            ds["time"] = pd.to_datetime(ds["time"].values)
        except Exception:
            pass

    var = find_temp_var(ds, candidates=("tas","air_temperature","t2m"))
    # Standardize but keep 2D target if it has no time
    ds = ensure_time_and_dims(ds, preferred_time="1979-01-01", target_var=var, keep_2d_target_if_no_time=True)

    try:
        mean_temp = mean_within_polygon(ds, var, gdf_india, time_idx=0)
    except Exception as e:
        print(f"  Error computing mean for {name}: {e}")
        mean_temp = np.nan

    results[name] = mean_temp
    ds.close()
    print(f"  {name}: mean temp = {mean_temp:.4f} °C")

# process extra files
for label, fp in extra_files.items():
    if not os.path.exists(fp):
        print(f"Warning: {fp} not found — skipping {label}")
        continue

    ds = xr.open_dataset(fp, decode_times=False)

    # try to convert time coord to datetimes if present
    if "time" in ds.coords and not np.issubdtype(ds["time"].dtype, np.datetime64):
        try:
            ds["time"] = pd.to_datetime(ds["time"].values)
        except Exception:
            pass

    var = find_temp_var(ds, candidates=("tas","air_temperature","t2m"))

    # For ensemble: if var has 'model' dim and you want per-member means,
    # you can compute them here. For now we'll average across model dimension.
    if "model" in ds[var].dims:
        # average across model first (remove model dim)
        ds = ds.copy()
        ds[var] = ds[var].mean(dim="model", skipna=True)

    ds = ensure_time_and_dims(ds, preferred_time="1979-01-01", target_var=var, keep_2d_target_if_no_time=True)

    try:
        mean_temp = mean_within_polygon(ds, var, gdf_india, time_idx=0)
    except Exception as e:
        print(f"  Error computing mean for {label}: {e}")
        mean_temp = np.nan

    results[label] = mean_temp
    ds.close()
    print(f"  {label}: mean temp = {mean_temp:.4f} °C")

# ------------------ Build results DataFrame and plot ------------------
df = pd.Series(results).rename("mean_temp_degC").sort_values(ascending=False).to_frame()
df.index.name = "dataset"
print("\nFinal results:")
print(df)

# Bar plot
plt.figure(figsize=(10,6))
ax = df["mean_temp_degC"].plot.bar(edgecolor='black')
ax.set_ylabel("Mean Temperature inside India (°C)", fontsize=12, fontweight='bold')
ax.set_title("Mean surface temperature inside India (one time slice per file)", fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
out_fig = os.path.join(output_dir, "mean_temp_by_dataset_india.png")
plt.savefig(out_fig, dpi=300)
print(f"\nPlot saved to: {out_fig}")
plt.show()

In [ ]:
#!/usr/bin/env python3
# Fast Dask-based pipeline: annual India-mean from daily NetCDF files (1979-2014)
# Includes Fix B: strict token-match for model tag (only accepts separators '_' or '.')

import os
import re
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import geopandas as gpd
import rasterio.features
import rasterio.transform
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt

# ---------------- CONFIG ----------------
model_base = "<SET YOUR PATH HERE>"
models = {
    "CESM2-WACCM": "1979-01-01_CESM2-WACCM.nc",
    "CESM2": "1979-01-01_CESM2.nc",
    "EC-Earth3-AerChem": "1979-01-01_EC-Earth3-AerChem.nc",
    "EC-Earth3-CC": "1979-01-01_EC-Earth3-CC.nc",
    "HadGEM3-GC31-LL": "1979-01-01_HadGEM3-GC31-LL.nc",
    "HadGEM3-GC31-MM": "1979-01-01_HadGEM3-GC31-MM.nc",
    "INM-CM4-8": "1979-01-01_INM-CM4-8.nc",
    "INM-CM5-0": "1979-01-01_INM-CM5-0.nc",
    "MPI-ESM1-2-HR": "1979-01-01_MPI-ESM1-2-HR.nc",
    "NorESM2-MM": "1979-01-01_NorESM2-MM.nc",
}
extra_files_dirs = {
    "Ensemble": "<SET YOUR PATH HERE>",
    "MSWX": "<SET YOUR PATH HERE>",
}
india_shp = "<SET YOUR PATH HERE>"
output_dir = "<SET YOUR PATH HERE>"
os.makedirs(output_dir, exist_ok=True)

START_YEAR = 1979
END_YEAR = 2014
YEARS = list(range(START_YEAR, END_YEAR + 1))

# ---------------- helpers ----------------
date_patterns = [
    (re.compile(r"(\d{4}-\d{2}-\d{2})"), "%Y-%m-%d"),
    (re.compile(r"(\d{8})"), "%Y%m%d"),
    (re.compile(r"(\d{7})"), "%Y%j"),  # YYYYDOY
]

def parse_date_from_filename(fname):
    base = os.path.basename(fname)
    for patt, fmt in date_patterns:
        m = patt.search(base)
        if m:
            s = m.group(1)
            try:
                return pd.to_datetime(s, format=fmt)
            except Exception:
                try:
                    return pd.to_datetime(s)
                except Exception:
                    continue
    m = re.search(r"(\d{4})", base)
    if m:
        return pd.to_datetime(f"{int(m.group(1))}-01-01")
    return None

def safe_read_country_shp(shp_path, country_name="India"):
    try:
        gdf = gpd.read_file(shp_path)
        if gdf.empty:
            raise RuntimeError("Empty geometry")
        print(f"Loaded local shapefile: {shp_path}")
        return gdf
    except Exception as e:
        print(f"Local shapefile not usable ({e}). Falling back to Natural Earth lowres.")
        world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
        gdf_country = world[world["name"].str.lower() == country_name.lower()]
        if gdf_country.empty:
            raise RuntimeError("Natural Earth does not contain the requested country.")
        return gdf_country

def get_model_tag_from_sample(sample_fname):
    base = os.path.basename(sample_fname)
    parts = base.split("_", 1)
    if len(parts) == 2:
        tag = parts[1].replace(".nc","")
        return tag
    m = re.search(r"[A-Za-z].*$", base)
    return m.group(0).replace(".nc","") if m else base.replace(".nc","")

# ---- Fix B: strict token-match allowing only '_' or '.' as separators ----
def files_for_model_token_sep(model_base, sample_fname, tag=None):
    """
    Return files whose filename contains the tag bounded by start/end or '_' or '.'.
    This prevents 'CESM2' matching 'CESM2-WACCM'.
    """
    if tag is None:
        tag = get_model_tag_from_sample(sample_fname)

    pattern1 = os.path.join(model_base, f"*_{tag}*.nc")
    pattern2 = os.path.join(model_base, f"*{tag}*.nc")
    candidates = sorted(set(glob.glob(pattern1) + glob.glob(pattern2)))

    # token must be bounded by start/end or underscore or dot
    token_re = re.compile(r'(^|[_\.])' + re.escape(tag) + r'($|[_\.])')

    filtered = [f for f in candidates if token_re.search(os.path.basename(f))]

    # If nothing matched (rare), fall back to candidates to avoid empty result
    return filtered or candidates

# Build a 2D binary mask (1 inside India) aligned to dataset lat/lon
def build_mask_for_ds(ds_sample, gdf_india):
    # ds_sample: dataarray with lat & lon 1D coords
    if "lat" not in ds_sample.coords or "lon" not in ds_sample.coords:
        raise RuntimeError("Dataset must have 1D lat and lon coords named 'lat'/'lon'.")
    lons = ds_sample["lon"].values
    lats = ds_sample["lat"].values
    minx, maxx = float(lons.min()), float(lons.max())
    miny, maxy = float(lats.min()), float(lats.max())
    width = int(lons.size)
    height = int(lats.size)
    transform = rasterio.transform.from_bounds(minx, miny, maxx, maxy, width, height)
    shapes = ((geom, 1) for geom in gdf_india.geometry)
    mask = rasterio.features.rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        all_touched=True,
        dtype=np.uint8
    )
    # flip rows if lat ascending
    if lats[0] < lats[-1]:
        mask = np.flipud(mask)
    mask_da = xr.DataArray(mask.astype(int), coords={"lat": lats, "lon": lons}, dims=("lat","lon"))
    return mask_da

def find_temp_var_in_ds(ds, candidates=("tas","air_temperature","t2m")):
    for v in candidates:
        if v in ds.data_vars:
            return v
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating) and len(ds[v].dims) >= 2:
            return v
    raise RuntimeError("No suitable temperature-like variable found in dataset.")

# Preprocess and open multiple daily files into a single lazy Dataset
def open_model_daily_files(files, var_candidates=("tas","air_temperature","t2m"), chunks=None):
    ds_list = []
    times = []
    for f in sorted(files):
        try:
            ds = xr.open_dataset(f, decode_times=False, mask_and_scale=True, use_cftime=False)
        except Exception as e:
            print(f"  WARNING: failed to open {f}: {e}")
            continue
        try:
            var = find_temp_var_in_ds(ds, candidates=var_candidates)
        except Exception:
            ds.close(); continue
        da = ds[var]
        if "time" in da.dims:
            da = da.mean(dim="time", skipna=True)
        if "model" in da.dims:
            da = da.mean(dim="model", skipna=True)
        # expand a time axis of length 1 for concatenation
        da = da.expand_dims(time=[0])
        ds_small = da.to_dataset(name=var)
        ds_list.append(ds_small)
        t = parse_date_from_filename(f)
        times.append(t if t is not None else np.datetime64('NaT'))
        ds.close()

    if not ds_list:
        return None

    combined = xr.concat(ds_list, dim="time")
    # assign parsed times (some may be NaT)
    combined = combined.assign_coords(time=("time", pd.to_datetime(times)))
    if chunks is not None:
        combined = combined.chunk(chunks)
    return combined

# ---------------- MAIN PROCESS ----------------
print("Loading India shapefile...")
gdf_india = safe_read_country_shp(india_shp, country_name="India")

all_results = {}
# tuning: adjust chunks if needed
DEFAULT_CHUNKS = {"time": 365, "lat": 100, "lon": 100}

for model_name, sample_fname in models.items():
    print(f"\nProcessing model: {model_name}")
    model_tag = get_model_tag_from_sample(sample_fname)
    files = files_for_model_token_sep(model_base, sample_fname, tag=model_tag)
    if not files:
        print(f"  No daily files found for model tag '{model_tag}'.")
        all_results[model_name] = pd.Series(index=YEARS, data=np.nan)
        continue

    print(f"  Found {len(files)} files, opening lazily with xarray...")
    ds = open_model_daily_files(files, chunks=DEFAULT_CHUNKS)
    if ds is None:
        print("  No readable files -> skipping")
        all_results[model_name] = pd.Series(index=YEARS, data=np.nan)
        continue

    var = list(ds.data_vars)[0]
    da = ds[var]

    print("  Building India mask for dataset grid...")
    mask_da = build_mask_for_ds(da.isel(time=0), gdf_india)
    mask_da = mask_da.astype(int).chunk({"lat": da.chunksizes.get("lat", da.sizes["lat"]),
                                         "lon": da.chunksizes.get("lon", da.sizes["lon"])})

    print("  Computing daily weighted means (lazy with Dask)...")
    if "lat" in da.coords:
        weights_da = xr.DataArray(np.cos(np.deg2rad(da["lat"])), coords={"lat": da["lat"]}, dims=["lat"])
        num = (da * weights_da).where(mask_da == 1)
        denom = weights_da.where(mask_da == 1)
        num_sum = num.sum(dim=("lat","lon"), skipna=True)
        denom_sum = denom.sum(dim=("lat","lon"), skipna=True)
        daily_mean = (num_sum / denom_sum)
    else:
        daily_mean = da.where(mask_da == 1).mean(dim=("lat","lon"), skipna=True)

    print("  Aggregating to annual means (lazy)...")
    try:
        if not np.issubdtype(daily_mean["time"].dtype, np.datetime64):
            daily_mean["time"] = pd.to_datetime(daily_mean["time"].values)
    except Exception:
        pass

    ann = daily_mean.groupby("time.year").mean(dim="time", skipna=True)
    print("  Triggering computation with Dask (this may take a while but is parallel)...")
    with ProgressBar():
        ann_pd = ann.compute().to_series()
    ann_pd = ann_pd.reindex(YEARS)
    all_results[model_name] = ann_pd

# Process extras (ensemble dir and MSWX dir) using the same token-based discovery for directories
for label, path in extra_files_dirs.items():
    print(f"\nProcessing extra dataset: {label}")
    if not os.path.exists(path):
        print(f"  Path not found: {path}; skipping.")
        all_results[label] = pd.Series(index=YEARS, data=np.nan)
        continue
    files = sorted(glob.glob(os.path.join(path, "*.nc")))
    if not files:
        print(f"  No .nc files in {path}")
        all_results[label] = pd.Series(index=YEARS, data=np.nan)
        continue
    ds = open_model_daily_files(files, chunks=DEFAULT_CHUNKS)
    if ds is None:
        all_results[label] = pd.Series(index=YEARS, data=np.nan)
        continue
    var = list(ds.data_vars)[0]
    da = ds[var]
    mask_da = build_mask_for_ds(da.isel(time=0), gdf_india)
    mask_da = mask_da.astype(int).chunk({"lat": da.chunksizes.get("lat", da.sizes["lat"]),
                                         "lon": da.chunksizes.get("lon", da.sizes["lon"])})
    if "lat" in da.coords:
        weights_da = xr.DataArray(np.cos(np.deg2rad(da["lat"])), coords={"lat": da["lat"]}, dims=["lat"])
        num = (da * weights_da).where(mask_da == 1)
        denom = weights_da.where(mask_da == 1)
        num_sum = num.sum(dim=("lat","lon"), skipna=True)
        denom_sum = denom.sum(dim=("lat","lon"), skipna=True)
        daily_mean = (num_sum / denom_sum)
    else:
        daily_mean = da.where(mask_da == 1).mean(dim=("lat","lon"), skipna=True)
    try:
        if not np.issubdtype(daily_mean["time"].dtype, np.datetime64):
            daily_mean["time"] = pd.to_datetime(daily_mean["time"].values)
    except Exception:
        pass
    ann = daily_mean.groupby("time.year").mean(dim="time", skipna=True)
    with ProgressBar():
        ann_pd = ann.compute().to_series()
    ann_pd = ann_pd.reindex(YEARS)
    all_results[label] = ann_pd

# Combine and save
df = pd.DataFrame(all_results)
csv_out = os.path.join(output_dir, "annual_mean_timeseries_india_1979_2014_fast_fixB.csv")
df.to_csv(csv_out, float_format="%.4f")
print(f"\nSaved CSV to: {csv_out}")

# Plot
plt.figure(figsize=(12,6))
for col in df.columns:
    plt.plot(df.index, df[col], label=col, linewidth=1.6)
plt.legend(ncol=2)
plt.xlabel("Year")
plt.ylabel("Annual mean temp inside India (°C)")
plt.title("Annual India-mean temperature (1979-2014) - fast Dask pipeline (Fix B)")
plt.grid(alpha=0.3)
out_png = os.path.join(output_dir, "annual_mean_timeseries_india_1979_2014_fast_fixB.png")
plt.tight_layout()
plt.savefig(out_png, dpi=300)
print(f"Saved plot to: {out_png}")
plt.show()

In [ ]:
#!/usr/bin/env python3
# Fast Dask-based pipeline: annual India-mean from daily NetCDF files (1979-2014)
# Includes Fix B: strict token-match for model tag (only accepts separators '_' or '.')
# Publication-quality plotting for Q1 journals

import os
import re
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import geopandas as gpd
import rasterio.features
import rasterio.transform
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ---------------- CONFIG ----------------
model_base = "<SET YOUR PATH HERE>"
models = {
    "CESM2-WACCM": "1979-01-01_CESM2-WACCM.nc",
    "CESM2": "1979-01-01_CESM2.nc",
    "EC-Earth3-AerChem": "1979-01-01_EC-Earth3-AerChem.nc",
    "EC-Earth3-CC": "1979-01-01_EC-Earth3-CC.nc",
    "HadGEM3-GC31-LL": "1979-01-01_HadGEM3-GC31-LL.nc",
    "HadGEM3-GC31-MM": "1979-01-01_HadGEM3-GC31-MM.nc",
    "INM-CM4-8": "1979-01-01_INM-CM4-8.nc",
    "INM-CM5-0": "1979-01-01_INM-CM5-0.nc",
    "MPI-ESM1-2-HR": "1979-01-01_MPI-ESM1-2-HR.nc",
    "NorESM2-MM": "1979-01-01_NorESM2-MM.nc",
}
extra_files_dirs = {
    "Ensemble": "<SET YOUR PATH HERE>",
    "MSWX": "<SET YOUR PATH HERE>",
}
india_shp = "<SET YOUR PATH HERE>"
output_dir = "<SET YOUR PATH HERE>"
os.makedirs(output_dir, exist_ok=True)

START_YEAR = 1979
END_YEAR = 2014
YEARS = list(range(START_YEAR, END_YEAR + 1))

# ---------------- helpers ----------------
date_patterns = [
    (re.compile(r"(\d{4}-\d{2}-\d{2})"), "%Y-%m-%d"),
    (re.compile(r"(\d{8})"), "%Y%m%d"),
    (re.compile(r"(\d{7})"), "%Y%j"),  # YYYYDOY
]

def parse_date_from_filename(fname):
    base = os.path.basename(fname)
    for patt, fmt in date_patterns:
        m = patt.search(base)
        if m:
            s = m.group(1)
            try:
                return pd.to_datetime(s, format=fmt)
            except Exception:
                try:
                    return pd.to_datetime(s)
                except Exception:
                    continue
    m = re.search(r"(\d{4})", base)
    if m:
        return pd.to_datetime(f"{int(m.group(1))}-01-01")
    return None

def safe_read_country_shp(shp_path, country_name="India"):
    try:
        gdf = gpd.read_file(shp_path)
        if gdf.empty:
            raise RuntimeError("Empty geometry")
        print(f"Loaded local shapefile: {shp_path}")
        return gdf
    except Exception as e:
        print(f"Local shapefile not usable ({e}). Falling back to Natural Earth lowres.")
        world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
        gdf_country = world[world["name"].str.lower() == country_name.lower()]
        if gdf_country.empty:
            raise RuntimeError("Natural Earth does not contain the requested country.")
        return gdf_country

def get_model_tag_from_sample(sample_fname):
    base = os.path.basename(sample_fname)
    parts = base.split("_", 1)
    if len(parts) == 2:
        tag = parts[1].replace(".nc","")
        return tag
    m = re.search(r"[A-Za-z].*$", base)
    return m.group(0).replace(".nc","") if m else base.replace(".nc","")

# ---- Fix B: strict token-match allowing only '_' or '.' as separators ----
def files_for_model_token_sep(model_base, sample_fname, tag=None):
    """
    Return files whose filename contains the tag bounded by start/end or '_' or '.'.
    This prevents 'CESM2' matching 'CESM2-WACCM'.
    """
    if tag is None:
        tag = get_model_tag_from_sample(sample_fname)

    pattern1 = os.path.join(model_base, f"*_{tag}*.nc")
    pattern2 = os.path.join(model_base, f"*{tag}*.nc")
    candidates = sorted(set(glob.glob(pattern1) + glob.glob(pattern2)))

    # token must be bounded by start/end or underscore or dot
    token_re = re.compile(r'(^|[_\.])' + re.escape(tag) + r'($|[_\.])')

    filtered = [f for f in candidates if token_re.search(os.path.basename(f))]

    # If nothing matched (rare), fall back to candidates to avoid empty result
    return filtered or candidates

# Build a 2D binary mask (1 inside India) aligned to dataset lat/lon
def build_mask_for_ds(ds_sample, gdf_india):
    # ds_sample: dataarray with lat & lon 1D coords
    if "lat" not in ds_sample.coords or "lon" not in ds_sample.coords:
        raise RuntimeError("Dataset must have 1D lat and lon coords named 'lat'/'lon'.")
    lons = ds_sample["lon"].values
    lats = ds_sample["lat"].values
    minx, maxx = float(lons.min()), float(lons.max())
    miny, maxy = float(lats.min()), float(lats.max())
    width = int(lons.size)
    height = int(lats.size)
    transform = rasterio.transform.from_bounds(minx, miny, maxx, maxy, width, height)
    shapes = ((geom, 1) for geom in gdf_india.geometry)
    mask = rasterio.features.rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        all_touched=True,
        dtype=np.uint8
    )
    # flip rows if lat ascending
    if lats[0] < lats[-1]:
        mask = np.flipud(mask)
    mask_da = xr.DataArray(mask.astype(int), coords={"lat": lats, "lon": lons}, dims=("lat","lon"))
    return mask_da

def find_temp_var_in_ds(ds, candidates=("tas","air_temperature","t2m")):
    for v in candidates:
        if v in ds.data_vars:
            return v
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating) and len(ds[v].dims) >= 2:
            return v
    raise RuntimeError("No suitable temperature-like variable found in dataset.")

# Preprocess and open multiple daily files into a single lazy Dataset
def open_model_daily_files(files, var_candidates=("tas","air_temperature","t2m"), chunks=None):
    ds_list = []
    times = []
    for f in sorted(files):
        try:
            ds = xr.open_dataset(f, decode_times=False, mask_and_scale=True, use_cftime=False)
        except Exception as e:
            print(f"  WARNING: failed to open {f}: {e}")
            continue
        try:
            var = find_temp_var_in_ds(ds, candidates=var_candidates)
        except Exception:
            ds.close(); continue
        da = ds[var]
        if "time" in da.dims:
            da = da.mean(dim="time", skipna=True)
        if "model" in da.dims:
            da = da.mean(dim="model", skipna=True)
        # expand a time axis of length 1 for concatenation
        da = da.expand_dims(time=[0])
        ds_small = da.to_dataset(name=var)
        ds_list.append(ds_small)
        t = parse_date_from_filename(f)
        times.append(t if t is not None else np.datetime64('NaT'))
        ds.close()

    if not ds_list:
        return None

    combined = xr.concat(ds_list, dim="time")
    # assign parsed times (some may be NaT)
    combined = combined.assign_coords(time=("time", pd.to_datetime(times)))
    if chunks is not None:
        combined = combined.chunk(chunks)
    return combined

# ---------------- MAIN PROCESS ----------------
print("Loading India shapefile...")
gdf_india = safe_read_country_shp(india_shp, country_name="India")

all_results = {}
# tuning: adjust chunks if needed
DEFAULT_CHUNKS = {"time": 365, "lat": 100, "lon": 100}

for model_name, sample_fname in models.items():
    print(f"\nProcessing model: {model_name}")
    model_tag = get_model_tag_from_sample(sample_fname)
    files = files_for_model_token_sep(model_base, sample_fname, tag=model_tag)
    if not files:
        print(f"  No daily files found for model tag '{model_tag}'.")
        all_results[model_name] = pd.Series(index=YEARS, data=np.nan)
        continue

    print(f"  Found {len(files)} files, opening lazily with xarray...")
    ds = open_model_daily_files(files, chunks=DEFAULT_CHUNKS)
    if ds is None:
        print("  No readable files -> skipping")
        all_results[model_name] = pd.Series(index=YEARS, data=np.nan)
        continue

    var = list(ds.data_vars)[0]
    da = ds[var]

    print("  Building India mask for dataset grid...")
    mask_da = build_mask_for_ds(da.isel(time=0), gdf_india)
    mask_da = mask_da.astype(int).chunk({"lat": da.chunksizes.get("lat", da.sizes["lat"]),
                                         "lon": da.chunksizes.get("lon", da.sizes["lon"])})

    print("  Computing daily weighted means (lazy with Dask)...")
    if "lat" in da.coords:
        weights_da = xr.DataArray(np.cos(np.deg2rad(da["lat"])), coords={"lat": da["lat"]}, dims=["lat"])
        num = (da * weights_da).where(mask_da == 1)
        denom = weights_da.where(mask_da == 1)
        num_sum = num.sum(dim=("lat","lon"), skipna=True)
        denom_sum = denom.sum(dim=("lat","lon"), skipna=True)
        daily_mean = (num_sum / denom_sum)
    else:
        daily_mean = da.where(mask_da == 1).mean(dim=("lat","lon"), skipna=True)

    print("  Aggregating to annual means (lazy)...")
    try:
        if not np.issubdtype(daily_mean["time"].dtype, np.datetime64):
            daily_mean["time"] = pd.to_datetime(daily_mean["time"].values)
    except Exception:
        pass

    ann = daily_mean.groupby("time.year").mean(dim="time", skipna=True)
    print("  Triggering computation with Dask (this may take a while but is parallel)...")
    with ProgressBar():
        ann_pd = ann.compute().to_series()
    ann_pd = ann_pd.reindex(YEARS)
    all_results[model_name] = ann_pd

# Process extras (ensemble dir and MSWX dir) using the same token-based discovery for directories
for label, path in extra_files_dirs.items():
    print(f"\nProcessing extra dataset: {label}")
    if not os.path.exists(path):
        print(f"  Path not found: {path}; skipping.")
        all_results[label] = pd.Series(index=YEARS, data=np.nan)
        continue
    files = sorted(glob.glob(os.path.join(path, "*.nc")))
    if not files:
        print(f"  No .nc files in {path}")
        all_results[label] = pd.Series(index=YEARS, data=np.nan)
        continue
    ds = open_model_daily_files(files, chunks=DEFAULT_CHUNKS)
    if ds is None:
        all_results[label] = pd.Series(index=YEARS, data=np.nan)
        continue
    var = list(ds.data_vars)[0]
    da = ds[var]
    mask_da = build_mask_for_ds(da.isel(time=0), gdf_india)
    mask_da = mask_da.astype(int).chunk({"lat": da.chunksizes.get("lat", da.sizes["lat"]),
                                         "lon": da.chunksizes.get("lon", da.sizes["lon"])})
    if "lat" in da.coords:
        weights_da = xr.DataArray(np.cos(np.deg2rad(da["lat"])), coords={"lat": da["lat"]}, dims=["lat"])
        num = (da * weights_da).where(mask_da == 1)
        denom = weights_da.where(mask_da == 1)
        num_sum = num.sum(dim=("lat","lon"), skipna=True)
        denom_sum = denom.sum(dim=("lat","lon"), skipna=True)
        daily_mean = (num_sum / denom_sum)
    else:
        daily_mean = da.where(mask_da == 1).mean(dim=("lat","lon"), skipna=True)
    try:
        if not np.issubdtype(daily_mean["time"].dtype, np.datetime64):
            daily_mean["time"] = pd.to_datetime(daily_mean["time"].values)
    except Exception:
        pass
    ann = daily_mean.groupby("time.year").mean(dim="time", skipna=True)
    with ProgressBar():
        ann_pd = ann.compute().to_series()
    ann_pd = ann_pd.reindex(YEARS)
    all_results[label] = ann_pd

# Combine and save
df = pd.DataFrame(all_results)
csv_out = os.path.join(output_dir, "annual_mean_timeseries_india_1979_2014_fast_fixB.csv")
df.to_csv(csv_out, float_format="%.4f")
print(f"\nSaved CSV to: {csv_out}")

# ---------------- PUBLICATION-QUALITY PLOT ----------------

# Set publication-quality parameters
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
rcParams['font.size'] = 11
rcParams['axes.labelsize'] = 12
rcParams['axes.titlesize'] = 13
rcParams['xtick.labelsize'] = 11
rcParams['ytick.labelsize'] = 11
rcParams['legend.fontsize'] = 9
rcParams['figure.dpi'] = 300
rcParams['savefig.dpi'] = 300
rcParams['lines.linewidth'] = 1.5
rcParams['axes.linewidth'] = 1.2
rcParams['grid.linewidth'] = 0.5

# Professional color palette (colorblind-friendly)
model_colors = {
    'CESM2-WACCM': '#1f77b4',      # Blue
    'CESM2': '#ff7f0e',             # Orange
    'EC-Earth3-AerChem': '#2ca02c', # Green
    'EC-Earth3-CC': '#d62728',      # Red
    'HadGEM3-GC31-LL': '#9467bd',   # Purple
    'HadGEM3-GC31-MM': '#8c564b',   # Brown
    'INM-CM4-8': '#e377c2',         # Pink
    'INM-CM5-0': '#7f7f7f',         # Gray
    'MPI-ESM1-2-HR': '#bcbd22',     # Yellow-green
    'NorESM2-MM': '#17becf',        # Cyan
}

# Markers for each model (distinct shapes)
model_markers = {
    'CESM2-WACCM': 'o',      # Circle
    'CESM2': 's',             # Square
    'EC-Earth3-AerChem': '^', # Triangle up
    'EC-Earth3-CC': 'v',      # Triangle down
    'HadGEM3-GC31-LL': 'D',   # Diamond
    'HadGEM3-GC31-MM': 'p',   # Pentagon
    'INM-CM4-8': '*',         # Star
    'INM-CM5-0': 'X',         # X
    'MPI-ESM1-2-HR': 'h',     # Hexagon
    'NorESM2-MM': 'd',        # Thin diamond
}

fig, ax = plt.subplots(figsize=(14, 7))

# Plot CMIP6 models first (thinner lines, with markers)
for col in df.columns:
    if col in model_colors:
        ax.plot(df.index, df[col], 
                color=model_colors[col],
                marker=model_markers[col],
                markersize=5,
                markevery=2,  # Show marker every 2 years to avoid clutter
                linewidth=1.2,
                alpha=0.7,
                label=col,
                zorder=2)

# Plot Ensemble with emphasis (thicker line, no marker clutter)
if 'Ensemble' in df.columns:
    ax.plot(df.index, df['Ensemble'],
            color='#000000',  # Black
            linewidth=3.0,
            linestyle='-',
            marker='o',
            markersize=6,
            markevery=3,
            label='Ensemble Mean',
            zorder=4,
            alpha=0.9)

# Plot MSWX (reference/observational) with highest emphasis
if 'MSWX' in df.columns:
    ax.plot(df.index, df['MSWX'],
            color='#C41E3A',  # Deep red for reference data
            linewidth=3.5,
            linestyle='-',
            marker='D',
            markersize=7,
            markevery=3,
            label='MSWX (Reference)',
            zorder=5,
            alpha=1.0)

# Styling
ax.set_xlabel('Year', fontsize=13, fontweight='bold')
ax.set_ylabel('Annual Mean Temperature (°C)', fontsize=13, fontweight='bold')
# ax.set_title('Annual India-Mean Temperature Time Series (1979–2014)\nCMIP6 Models vs. Reference Dataset',
#              fontsize=14, fontweight='bold', pad=15)

# Grid
ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.3, zorder=1)
ax.set_axisbelow(True)

# Legend - organize into two columns with reference data separate
handles, labels = ax.get_legend_handles_labels()

# Reorder: MSWX and Ensemble first, then models
priority_labels = ['MSWX (Reference)', 'Ensemble Mean']
priority_handles = []
priority_label_list = []
model_handles = []
model_labels = []

for h, l in zip(handles, labels):
    if l in priority_labels:
        priority_handles.append(h)
        priority_label_list.append(l)
    else:
        model_handles.append(h)
        model_labels.append(l)

# Combine with priority items first
all_handles = priority_handles + model_handles
all_labels = priority_label_list + model_labels

legend = ax.legend(all_handles, all_labels,
                   loc='upper left',
                   ncol=3,
                   frameon=True,
                   fancybox=False,
                   shadow=False,
                   framealpha=0.95,
                   edgecolor='black',
                   columnspacing=1.0,
                   handlelength=2.5)
legend.get_frame().set_linewidth(1.2)

# Set axis limits with some padding
ax.set_xlim(df.index.min() - 1, df.index.max() + 1)
y_min, y_max = df.min().min(), df.max().max()
y_range = y_max - y_min
ax.set_ylim(y_min - 0.1 * y_range, y_max + 0.1 * y_range)

# Spine styling
for spine in ax.spines.values():
    spine.set_linewidth(1.2)
    spine.set_color('black')

# Tight layout
plt.tight_layout()

# Save in multiple formats for journal submission
out_png = os.path.join(output_dir, "annual_mean_timeseries_india_1979_2014_publication.png")
out_pdf = os.path.join(output_dir, "annual_mean_timeseries_india_1979_2014_publication.pdf")
out_eps = os.path.join(output_dir, "annual_mean_timeseries_india_1979_2014_publication.eps")

plt.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(out_pdf, bbox_inches='tight', facecolor='white')
plt.savefig(out_eps, bbox_inches='tight', facecolor='white', format='eps')

print(f"\nSaved publication-quality plots:")
print(f"  PNG: {out_png}")
print(f"  PDF: {out_pdf}")
print(f"  EPS: {out_eps}")

plt.show()

# ---------------- STATISTICS SUMMARY TABLE ----------------
print("\n" + "="*60)
print("STATISTICS SUMMARY (1979-2014)")
print("="*60)

# Calculate trends properly
def calculate_trend(series):
    """Calculate linear trend in °C/decade"""
    valid_data = series.dropna()
    if len(valid_data) > 2:
        # Use the index values where data is valid
        x = valid_data.index.values
        y = valid_data.values
        trend_per_year = np.polyfit(x, y, 1)[0]
        return trend_per_year * 10  # Convert to per decade
    else:
        return np.nan

stats_summary = pd.DataFrame({
    'Mean (°C)': df.mean(),
    'Std Dev (°C)': df.std(),
    'Min (°C)': df.min(),
    'Max (°C)': df.max(),
    'Trend (°C/decade)': [calculate_trend(df[col]) for col in df.columns]
})

# Sort with reference data first
priority_order = ['MSWX', 'Ensemble']
other_models = [col for col in stats_summary.index if col not in priority_order]
ordered_index = [col for col in priority_order if col in stats_summary.index] + sorted(other_models)
stats_summary = stats_summary.loc[ordered_index]

print(stats_summary.to_string(float_format='%.3f'))

stats_csv = os.path.join(output_dir, "statistics_summary_1979_2014.csv")
stats_summary.to_csv(stats_csv, float_format="%.4f")
print(f"\nStatistics saved to: {stats_csv}")
print("="*60)

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

# assume `combined` already exists (the xarray.Dataset you printed)
# combined: <xarray.Dataset> with dims (time: N, lat: 500, lon: 500)

# ----------------------------
# 1. Build a robust key extractor for mixed time types
# ----------------------------
def time_tuple(t):
    """
    Return (year, month, day, hour, minute, second) for a time object.
    Works for pandas.Timestamp, numpy.datetime64, cftime.* and strings.
    """
    # numpy datetime64 -> convert via pandas
    try:
        # direct attributes for pandas.Timestamp or cftime
        yr = int(t.year)
        mo = int(t.month)
        day = int(t.day)
        hr = int(getattr(t, "hour", 0) or 0)
        mn = int(getattr(t, "minute", 0) or 0)
        sc = int(getattr(t, "second", 0) or 0)
        return (yr, mo, day, hr, mn, sc)
    except Exception:
        # fallback: convert to string then parse with pandas
        s = str(t)
        try:
            ts = pd.to_datetime(s)
            return (ts.year, ts.month, ts.day, ts.hour, ts.minute, ts.second)
        except Exception:
            # as a last resort, return a tuple of strings (keeps a deterministic order)
            return tuple(s.split())

# Create list of time objects from the dataset
times = combined["time"].values  # likely an array of mixed types

# Build list of tuples (fast, pure-Python)
time_keys = [time_tuple(t) for t in times]

# Get sorting order (stable)
order = np.array(sorted(range(len(time_keys)), key=lambda i: time_keys[i]), dtype=int)

# If already sorted, this will be identity; otherwise reorder
if not np.all(order == np.arange(len(order))):
    combined = combined.isel(time=order)

print("Dataset sorted along time (mixed-type safe).")

# ----------------------------
# 2. Create explicit 'year' coordinate (int) to avoid calendar issues
# ----------------------------
years = np.array([k[0] if isinstance(k, tuple) and isinstance(k[0], (int, np.integer)) else int(pd.to_datetime(str(t)).year)
                  for k, t in zip(time_keys, times)], dtype=int)

# Reassign in dataset
combined = combined.assign_coords(year=("time", years))
print("Assigned explicit integer 'year' coordinate.")

# Quick sanity check: show unique years and counts
(unique_years, counts) = np.unique(years, return_counts=True)
print("Years present:", unique_years)
print("Counts per year (example):", dict(zip(unique_years[:5], counts[:5])))

# ----------------------------
# 3. Compute annual mean using the explicit 'year' coordinate
# ----------------------------
# Use the 'tas' variable; adapt if your variable name is different
varname = "tas"
if varname not in combined.data_vars:
    raise KeyError(f"Variable '{varname}' not found in dataset. Available: {list(combined.data_vars)}")

annual_mean = combined[varname].groupby("year").mean(dim="time")  # lazy (dask-backed) if arrays are chunked

# Wrap into a Dataset and save to NetCDF
out_nc = Path("annual_mean_fixed_time.nc")
annual_ds = annual_mean.to_dataset(name=f"{varname}_annual_mean")

encoding = {f"{varname}_annual_mean": {"zlib": True, "complevel": 4}}
annual_ds.to_netcdf(out_nc, mode="w", encoding=encoding)
print(f"Saved annual means to {out_nc.resolve()}")

# ----------------------------
# 4. Plot per-year maps (reopen saved file to keep memory low)
# ----------------------------
plot_dir = Path("annual_mean_plots_fixed_time")
plot_dir.mkdir(exist_ok=True)

ds_ann = xr.open_dataset(out_nc, chunks={"lat": 50, "lon": 50})
# If `year` is a coordinate in the file, use it; else use ds_ann.indexes or variable
if "year" in ds_ann.coords:
    plot_years = ds_ann["year"].values
else:
    # fallback: use the dimension name of the annual variable (should be 'year')
    plot_years = ds_ann[f"{varname}_annual_mean"].coords[annual_mean.dims[0]].values

for y in plot_years:
    da = ds_ann[f"{varname}_annual_mean"].sel(year=int(y))
    plt.figure(figsize=(8, 5))
    da.plot(cbar_kwargs={"label": f"{varname} (units)"})
    plt.title(f"Annual mean {varname} - {int(y)}")
    plt.tight_layout()
    fn = plot_dir / f"annual_mean_{int(y)}.png"
    plt.savefig(fn, dpi=150)
    plt.close()

print(f"Saved plots to {plot_dir.resolve()}")

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import glob
from tqdm import tqdm
import gc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.ops import unary_union
import shapely.geometry as sgeom
from shapely.prepared import prep
import re

def extract_year_from_filename(filepath):
    """
    Extract year from filename.
    Assumes filename contains YYYY pattern (e.g., 2005, 2014, etc.)
    """
    filename = Path(filepath).name
    match = re.search(r'(\d{4})', filename)
    if match:
        return int(match.group(1))
    return None

def is_year_in_range(filepath, start_year=2005, end_year=2014):
    """Check if file's year is within the specified range"""
    year = extract_year_from_filename(filepath)
    if year is None:
        return False
    return start_year <= year <= end_year

def clip_data_to_india(data_array, lat, lon):
    """
    Clip data array to India boundary shapefile - OPTIMIZED VERSION
    
    Key improvements for ~10x speedup:
    ✅ Uses shapely.prepared.prep() for fast spatial indexing
    ✅ Progress bar shows clipping progress
    ✅ Minimal overhead in point containment checks
    """
    shapefile_path = "<SET YOUR PATH HERE>"
    
    try:
        print("   Clipping data to India boundary...")
        
        # Load shapefile
        gdf = gpd.read_file(shapefile_path)
        boundary = unary_union(gdf.geometry)
        
        # OPTIMIZATION 1: Prepare geometry for ~10x faster containment checks
        # PreparedGeometry uses spatial indexing internally
        prepared_boundary = prep(boundary)
        
        # Create a mask for points inside India
        lon_grid, lat_grid = np.meshgrid(lon, lat)
        points = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])
        
        # OPTIMIZATION 2: Use prepared geometry with progress bar
        # Progress bar shows you which points are being checked (can cancel if needed)
        print("   Checking point containment...")
        mask = np.array([
            prepared_boundary.contains(sgeom.Point(p)) 
            for p in tqdm(points, desc="     Points checked", leave=False, unit="pt", ncols=60)
        ])
        mask = mask.reshape(lon_grid.shape)
        
        # Apply mask to data - set values outside India to NaN
        clipped_data = data_array.copy()
        clipped_data[~mask] = np.nan
        
        valid_cells = np.sum(~np.isnan(clipped_data))
        print(f"   ✓ Data clipped - {valid_cells:,} valid cells in India")
        
        return clipped_data, boundary
    except Exception as e:
        print(f"   Warning: Could not clip to India boundary: {e}")
        print("   Using full data without clipping")
        return data_array, None


def clip_data_to_india_rasterized(data_array, lat, lon):
    """
    Alternative clipping using rasterization - even faster for complex geometries.
    Requires: rasterio, rasterize
    
    Typically 2-3x faster than prepared geometry approach.
    """
    shapefile_path = "<SET YOUR PATH HERE>"
    
    try:
        from rasterio.features import rasterize
        from rasterio.transform import from_bounds
        
        print("   Clipping data to India boundary (rasterized)...")
        
        # Load shapefile
        gdf = gpd.read_file(shapefile_path)
        
        # Create rasterization transform
        lon_min, lon_max = lon.min(), lon.max()
        lat_min, lat_max = lat.min(), lat.max()
        
        transform = from_bounds(lon_min, lat_min, lon_max, lat_max, len(lon), len(lat))
        
        # Rasterize the geometry to a mask
        shapes = [(geom, 1) for geom in gdf.geometry]
        mask = rasterize(shapes, out_shape=data_array.shape, transform=transform, dtype=np.uint8)
        
        # Apply mask to data
        clipped_data = data_array.copy()
        clipped_data[mask == 0] = np.nan
        
        valid_cells = np.sum(~np.isnan(clipped_data))
        print(f"   ✓ Data clipped - {valid_cells} valid cells in India")
        
        return clipped_data, None
    except ImportError:
        print("   Warning: rasterio not installed. Using prepared geometry method.")
        # Fall back to prepared geometry version
        gdf = gpd.read_file(shapefile_path)
        boundary = unary_union(gdf.geometry)
        prepared_boundary = prep(boundary)
        
        lon_grid, lat_grid = np.meshgrid(lon, lat)
        points = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])
        mask = np.array([prepared_boundary.contains(sgeom.Point(p)) for p in tqdm(points, desc="     Points", leave=False)])
        mask = mask.reshape(lon_grid.shape)
        
        clipped_data = data_array.copy()
        clipped_data[~mask] = np.nan
        return clipped_data, None


# Define directories for each dataset
datasets_config = {
    'Raw': {
        'dir': "<SET YOUR PATH HERE>",
        'pattern': "*_CESM2.nc",
        'var_name': 'tas',
    },
    'QDM BC': {
        'dir': "<SET YOUR PATH HERE>",
        'pattern': "*_CESM2.nc",
        'var_name': 'tas',
    },
    'DDPM': {
        'dir': "<SET YOUR PATH HERE>",
        'pattern': "*_CESM2.nc",
        'var_name': 'tas',
    },
    'MSWX': {
        'dir': "<SET YOUR PATH HERE>",
        'pattern': "*.nc",
        'var_name': 'air_temperature',
    }
}

def compute_mean_dataset_memory_efficient(dataset_name, config, start_year=2005, end_year=2014, batch_size=100):
    """
    Compute mean for a dataset with optimized memory management.
    Filters files to specified year range.
    Uses float32 internally and only converts to float64 for final division.
    """
    
    base_dir = config['dir']
    pattern = config['pattern']
    var_name = config['var_name']
    
    # Get all files
    all_files = sorted(glob.glob(str(Path(base_dir) / pattern)))
    
    # Filter files by year range
    file_list = [f for f in all_files if is_year_in_range(f, start_year, end_year)]
    
    if not file_list:
        print(f"   No files found for {dataset_name} in period {start_year}-{end_year}")
        print(f"   (Total files available: {len(all_files)})")
        return None, 0, None, None
    
    print(f"   Found {len(file_list)} files in {start_year}-{end_year} (out of {len(all_files)} total)")
    
    # Initialize accumulators with float32 for memory efficiency
    running_sum = None
    total_count = 0
    total_errors = 0
    lat_values = None
    lon_values = None
    
    # Calculate total batches
    total_batches = (len(file_list) - 1) // batch_size + 1
    
    # Process files in batches with progress bar
    for batch_num in tqdm(range(total_batches), desc=f"   {dataset_name}", unit="batch", leave=False):
        batch_start = batch_num * batch_size
        batch_end = min(batch_start + batch_size, len(file_list))
        batch_files = file_list[batch_start:batch_end]
        
        batch_sum = None
        batch_count = 0
        
        # Process each file in the batch
        for filepath in batch_files:
            try:
                with xr.open_dataset(filepath, chunks={'time': 1}) as ds:
                    data_var = ds[var_name]
                    
                    # Extract spatial data (remove time dimension)
                    data_spatial = data_var.mean(dim='time')
                    
                    # Check shape
                    if data_spatial.shape != (500, 500):
                        continue
                    
                    # Keep as float32 for memory efficiency
                    data_values = data_spatial.values.astype(np.float32)
                    
                    # Extract coordinates on first file
                    if lat_values is None:
                        if 'lat' in ds.coords:
                            lat_values = ds['lat'].values
                        else:
                            lat_values = ds.coords[list(ds.coords.keys())[0]].values
                        
                        if 'lon' in ds.coords:
                            lon_values = ds['lon'].values
                        else:
                            lon_values = ds.coords[list(ds.coords.keys())[1]].values
                    
                    # Accumulate within batch
                    if batch_sum is None:
                        batch_sum = data_values.copy()
                    else:
                        batch_sum = batch_sum + data_values
                    
                    batch_count += 1
                    total_count += 1
            
            except Exception as e:
                total_errors += 1
                continue
        
        # Add batch sum to running total
        if batch_sum is not None:
            if running_sum is None:
                running_sum = batch_sum.copy()
            else:
                running_sum = running_sum + batch_sum
        
        # Explicitly clean up batch memory
        batch_sum = None
        gc.collect()
    
    if running_sum is None:
        print(f"   ERROR: No data accumulated for {dataset_name}")
        return None, total_count, lat_values, lon_values
    
    # Compute final mean (convert to float64 only for division if needed)
    mean_data = running_sum.astype(np.float64) / total_count
    
    print(f"   ✓ Successfully processed {total_count} files")
    
    return mean_data, total_count, lat_values, lon_values

def compute_differences_with_mswx(datasets, file_counts):
    """
    Compute differences while keeping memory footprint minimal.
    Processes difference calculations sequentially instead of storing all in memory.
    """
    
    differences = {}
    stats = {}
    
    if 'MSWX' not in datasets:
        print("ERROR: MSWX data not found")
        return differences, stats
    
    mswx_data = datasets['MSWX']['mean']
    
    for dataset_name in ['Raw', 'QDM BC', 'DDPM']:
        if dataset_name not in datasets:
            continue
        
        print(f"\nComputing difference: {dataset_name} - MSWX")
        
        dataset_values = datasets[dataset_name]['mean']
        
        # Compute difference (stays in memory only as needed)
        diff = dataset_values - mswx_data
        
        # Create xarray DataArray
        diff_da = xr.DataArray(
            diff,
            coords={'lat': datasets[dataset_name]['lat'], 'lon': datasets[dataset_name]['lon']},
            dims=['lat', 'lon'],
            name=f'{dataset_name}_diff'
        )
        differences[dataset_name] = diff_da
        
        # Compute and store statistics only (not full arrays)
        valid_diff = diff[~np.isnan(diff)]
        
        stats[dataset_name] = {
            'min': np.nanmin(diff),
            'max': np.nanmax(diff),
            'mean': np.nanmean(diff),
            'median': np.nanmedian(diff),
            'std': np.nanstd(diff),
            'rmse': np.sqrt(np.nanmean(diff**2)),
            'valid_cells': len(valid_diff)
        }
        
        print(f"   Min difference:    {stats[dataset_name]['min']:.4f}°C")
        print(f"   Max difference:    {stats[dataset_name]['max']:.4f}°C")
        print(f"   Mean difference: {stats[dataset_name]['mean']:.4f}°C")
        print(f"   Std deviation:   {stats[dataset_name]['std']:.4f}°C")
        
        # Clean up to free memory
        del diff
        gc.collect()
    
    return differences, stats

# Main execution
print("="*80)
print("Computing Difference Maps: Dataset vs MSWX Reference (2005-2014)")
print("Memory-Efficient Version with Optimized India Boundary Clipping")
print("="*80 + "\n")

datasets = {}
file_counts = {}

# MODIFIED: Specify year range 2005-2014
START_YEAR = 2005
END_YEAR = 2014

for dataset_name, config in datasets_config.items():
    print(f"Processing {dataset_name}...")
    mean_data, count, lat, lon = compute_mean_dataset_memory_efficient(
        dataset_name, config, start_year=START_YEAR, end_year=END_YEAR, batch_size=100
    )
    if mean_data is not None:
        datasets[dataset_name] = {
            'mean': mean_data,
            'lat': lat,
            'lon': lon
        }
        file_counts[dataset_name] = count
    print()
    gc.collect()  # Force garbage collection between datasets

# Compute differences
print("\n" + "="*80)
differences, stats = compute_differences_with_mswx(datasets, file_counts)

# Clip all difference data to India boundary using optimized method
print("\n" + "="*80)
print("Clipping all data to India boundary (OPTIMIZED)...")
print("="*80 + "\n")

india_boundary = None
india_bounds = None
for dataset_name in ['Raw', 'QDM BC', 'DDPM']:
    if dataset_name in differences:
        print(f"Clipping {dataset_name}...")
        clipped_data, india_boundary = clip_data_to_india(
            differences[dataset_name].values,
            differences[dataset_name].lat.values,
            differences[dataset_name].lon.values
        )
        
        # Update difference data array with clipped data
        differences[dataset_name].values = clipped_data
        
        # Get India boundary extent for focused plotting
        if india_boundary is not None and india_bounds is None:
            minx, miny, maxx, maxy = india_boundary.bounds
            india_bounds = [minx, maxx, miny, maxy]

# Create difference plots
fig = plt.figure(figsize=(22, 9))

# --- MODIFICATION: Adjust gridspec to make room for bottom colorbar ---
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], hspace=0.3, wspace=0.35,
                      top=0.9, bottom=0.2) # Make space at bottom
# --- End MODIFICATION ---

plot_titles = [
    'Raw CESM2 - MSWX\n(Bilinearly interpolated: 1.0° → 0.1°)',
    'QDM Bias Corrected CESM2 - MSWX\n(0.1° resolution)',
    'DDPM CESM2 - MSWX\n(Downscaled: 1.0° → 0.1°)',
]
dataset_names = ['Raw', 'QDM BC', 'DDPM']

# Define two different symmetric color limits
# 1. For Raw and QDM BC (larger scale)
vmin_rq = -10.0
vmax_rq = 10.0

# 2. For DDPM (much smaller scale, based on your stats: Min -0.23, Max 0.37)
vmin_ddpm = -2.5
vmax_ddpm = 2.5

# Plot each difference map
for plot_idx, (title, dataset_name) in enumerate(zip(plot_titles, dataset_names)):
    ax = fig.add_subplot(gs[0, plot_idx], projection=ccrs.PlateCarree())
    
    if dataset_name in differences:
        diff_data = differences[dataset_name]
        
        # Apply conditional vmin/vmax
        if dataset_name in ['Raw', 'QDM BC']:
            current_vmin = vmin_rq
            current_vmax = vmax_rq
        elif dataset_name == 'DDPM':
            current_vmin = vmin_ddpm
            current_vmax = vmax_ddpm
        
        # Plot with the correct vmin/vmax and a diverging colormap
        im = diff_data.plot(ax=ax, cmap='RdBu_r', vmin=current_vmin, vmax=current_vmax, 
                            add_colorbar=False, transform=ccrs.PlateCarree())
        
        # Add India boundary if available
        if india_boundary is not None:
            from cartopy.feature import ShapelyFeature
            india_feature = ShapelyFeature([india_boundary], ccrs.PlateCarree(), 
                                           facecolor='none', edgecolor='black', linewidth=1.5)
            ax.add_feature(india_feature)
        
        # Focus on India extent
        if india_bounds is not None:
            buffer = 1.0
            extent = [
                india_bounds[0] - buffer,  # min longitude
                india_bounds[1] + buffer,  # max longitude
                india_bounds[2] - buffer,  # min latitude
                india_bounds[3] + buffer   # max latitude
            ]
            ax.set_extent(extent, crs=ccrs.PlateCarree())
        
        ax.set_aspect('equal', adjustable='box')
        ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
        ax.set_xlabel('Longitude (°E)', fontsize=11)
        ax.set_ylabel('Latitude (°N)', fontsize=11)
        ax.gridlines(draw_labels=True, alpha=0.3, linestyle='--', linewidth=0.5)

# --- MODIFICATION: Add one HORIZONTAL (bottom) and one VERTICAL colorbar ---

# Adjust main plot area to make space on the right for vertical cbar
fig.subplots_adjust(right=0.9)

# Colorbar 1: For Raw and QDM BC (HORIZONTAL - at the bottom)
# Position: [left, bottom, width, height]
# Get position of first two axes to align it
ax1_pos = fig.axes[0].get_position()
ax2_pos = fig.axes[1].get_position()
cbar_ax_rq = fig.add_axes([ax1_pos.x0, 0.1, (ax2_pos.x1 - ax1_pos.x0), 0.025]) # Spans first two plots
sm_rq = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=vmin_rq, vmax=vmax_rq))
sm_rq.set_array([])
cbar_rq = fig.colorbar(sm_rq, cax=cbar_ax_rq, extend='both', orientation='horizontal')
cbar_rq.set_label('Temp. Difference (°C) (Raw & QDM BC)', fontsize=10, fontweight='bold')


# Colorbar 2: For DDPM (VERTICAL - at the right)
# Position: [left, bottom, width, height]
# We get the position of the 3rd axes to align it
ax3_pos = fig.axes[2].get_position()
cbar_ax_ddpm = fig.add_axes([0.92, ax3_pos.y0, 0.015, ax3_pos.height])
sm_ddpm = plt.cm.ScalarMappable(cmap='Spectral_r', norm=plt.Normalize(vmin=vmin_ddpm, vmax=vmax_ddpm))
sm_ddpm.set_array([])
cbar_ddpm = fig.colorbar(sm_ddpm, cax=cbar_ax_ddpm, extend='both', orientation='vertical')
cbar_ddpm.set_label('Temp. Difference (°C)\n(DDPM)', fontsize=10, fontweight='bold')
# --- End MODIFICATION ---

# Adjust suptitle to be at the top
plt.suptitle('Temperature Difference with Respect to MSWX Reference (2005-2014)\n', 
             fontsize=16, fontweight='bold', y=0.98)
             
# plt.savefig('/mnt/user-data/outputs/all_datasets_difference_india_clipped_2005_2014.png', dpi=150, bbox_inches='tight')
# print("Difference plot saved as 'all_datasets_difference_india_clipped_2005_2014.png'\n")
plt.show()

# Print summary statistics
print("="*80)
print(f"DIFFERENCE STATISTICS (Dataset - MSWX) - CLIPPED TO INDIA BOUNDARY ({START_YEAR}-{END_YEAR})")
print("="*80 + "\n")

for dataset_name in dataset_names:
    if dataset_name in differences:
        diff = differences[dataset_name].values
        valid_diff = diff[~np.isnan(diff)]
        
        print(f"{dataset_name}:")
        print(f"   Files used:         {file_counts[dataset_name]}")
        print(f"   Period:             {START_YEAR}-{END_YEAR}")
        print(f"   Shape:              (500, 500)")
        print(f"   Valid cells in India: {len(valid_diff)}")
        print(f"   Min difference:     {np.nanmin(diff):.4f}°C")
        print(f"   Max difference:     {np.nanmax(diff):.4f}°C")
        print(f"   Mean difference:    {np.nanmean(diff):.4f}°C")
        print(f"   Median difference:  {np.nanmedian(diff):.4f}°C")
        print(f"   Std Dev:            {np.nanstd(diff):.4f}°C")
        print(f"   RMSE:               {np.sqrt(np.nanmean(diff**2)):.4f}°C")
        print()

print("="*80)

# Final cleanup
gc.collect()